# 09 เก็บข้อมูลภายนอกจริงและหลักฐานวันที่ (v2 – งานวิจัยระหว่างดำเนินการ)

**เป้าหมาย:** (1) ตรวจหลักฐานรูปแบบวันที่รายปีจาก XLSX ทางการของ MOT ที่มีเซลล์วันที่แบบ typed โดยจับคู่ด้วย ACC_CODE เดิมแบบตรงตัวเท่านั้น (2) เก็บ/ตรวจแหล่งข้อมูลภายนอกจริงสำหรับ v2: ปริมาณจราจร AADT ของกรมทางหลวง, ขอบเขตจังหวัดและจุดสอบถามสภาพอากาศ, พยากรณ์อากาศย้อนหลังที่เก็บถาวร (JMA GSM, ล่วงหน้า 48 ชม.), ปฏิทินวันหยุดธนาคารแห่งประเทศไทย (3) ทำตารางความพร้อมของ feature ทุกกลุ่ม (เทศกาล, อากาศ, ฤดูกาล, ช่วงเวลา, จราจร, ประวัติ)

**ไม่ทำในขั้นนี้:** ไม่สร้าง feature panel, ไม่ fit/เทียบโมเดล, ไม่อ้างว่าชนะ baseline, ไม่แก้ข้อมูล/รายงาน/notebook/config ของ v1 (01–08)

**อินพุต:** `config/external_sources_v2.json`; snapshot ดิบ v1 `data/raw/accidents/*.csv` (อ่านอย่างเดียว ตรวจ SHA256 กับ `source_manifest.json`); ไฟล์ preflight ที่ Codex ดาวน์โหลดไว้ใน `data/external_v2/evidence/` (อ่านอย่างเดียว บันทึก hash และ mtime โดยไม่อ้างเวลาดาวน์โหลดเดิม); การดาวน์โหลด HTTPS จริงจากโฮสต์ใน allowlist (ไม่มี credential ไม่มีบริการเสียเงิน)

**เอาต์พุต:** ข้อมูลดิบ/แคช/ตารางรายแถว (ไม่อยู่ใน git) ใต้ `ROADAWARE_V2_EXTERNAL_ROOT` (ค่าเริ่มต้น `backend/data/external_v2/`: `registry.json`, `raw/…`, `derived/…`); รายงานขนาดเล็กใต้ `ROADAWARE_V2_OUTPUT_ROOT/reports/v2/sources/`: `sources_manifest.json`, `date_evidence_by_year.csv/json`, `external_schema_coverage.csv`, `feature_source_readiness.csv`, `traffic_resource_metadata.csv`, `province_points.csv`, `weather_coverage.csv`, `calendar_records.csv`, `source_audit_th.md`

**การตรวจ:** แคชใช้ซ้ำได้เฉพาะเมื่อ SHA256 ตรงกับ registry (ไม่ตรง = หยุดชัดเจน ไม่เขียนทับ); แหล่งที่ล้มเหลวถูกบันทึกเป็น unavailable ไม่ปลอมข้อมูล; วันที่ resolve ได้เฉพาะแถวที่ ID ตรงและวันที่ typed เห็นพ้องกันโดยไม่กำกวม; อากาศรวมรายวัน/6 ชม. ต้องครบชั่วโมง (ไม่รวมค่าว่างเป็น 0) และผ่านเงื่อนไขเวลาออกพยากรณ์; จุดจังหวัดต้องอยู่ในรูปจังหวัด; JSON เป็น strict (ไม่มี NaN)

**ข้อจำกัด:** notebook นี้รันหลายนาที (อากาศ 77 จังหวัด × 5 ปี แบ่ง chunk และหยุดเมื่อโดน rate limit แล้วรันต่อได้); AADT เป็นค่าเฉลี่ยรายปี ณ จุดสำรวจ ไม่ใช่จราจร real-time และรวมกันเป็นจำนวนรถไม่ได้; เวลาใน catalog เป็นเพียง proxy ของวันที่เผยแพร่; ระยะ lead ของ API ไม่ได้พิสูจน์ความหน่วงการเผยแพร่จริงในอดีต; วันหยุด ธปท. ไม่ใช่วันหยุดราชการทั้งหมด; ขอบเขต geoBoundaries มาจาก OSM ไม่ใช่ข้อมูลทางการของรัฐ

**รันอย่างไร:** VS Code เลือก kernel `.venv` แล้ว *Run All* (เปิดจาก root หรือโฟลเดอร์ notebooks ก็ได้) หรือ headless: `.venv/bin/python backend/scripts/run_v2_notebooks.py --stage collect`

In [ ]:
# FUNCTIONS (notebook 09) - self-contained; tests execute this cell directly (no network in tests)
import datetime as dt, hashlib, html as htmllib, io, json, math, os, re, socket, time, zipfile
from pathlib import Path
from urllib.parse import urlencode, urljoin, urlparse
from urllib.request import HTTPRedirectHandler, Request, build_opener
from urllib.error import HTTPError, URLError
import numpy as np, pandas as pd

EXCEL_ORIGIN = dt.date(1899, 12, 30)
TYPED_KINDS = ('typed_datetime', 'typed_date')
THAI_MONTHS = {'มกราคม': 1, 'กุมภาพันธ์': 2, 'มีนาคม': 3, 'เมษายน': 4, 'พฤษภาคม': 5, 'มิถุนายน': 6,
               'กรกฎาคม': 7, 'สิงหาคม': 8, 'กันยายน': 9, 'ตุลาคม': 10, 'พฤศจิกายน': 11, 'ธันวาคม': 12}
THAI_DIGITS = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
VERIFIED_FESTIVAL = 'verified_recurring_fixed_festival_in_official_source'
EXCLUDED_TEXT = 'excluded_unreviewed_text_extraction'
CAL_COLUMNS = ['date', 'name', 'festival_id', 'status', 'record_type', 'is_national_public_holiday', 'source_id', 'source_url',
               'source_sha256', 'source_year', 'announcement_date', 'announcement_date_status', 'source_text']
POINT_COLUMNS = ['province_code', 'name_th', 'name_en', 'shape_name', 'shape_iso', 'shape_id', 'shape_index', 'join_method', 'lat', 'lon',
                 'point_within_polygon', 'geometry_valid', 'in_v1_panel', 'point_method', 'point_meaning', 'geometry_source',
                 'geometry_sha256', 'province_table_sha256']


# ---------- generic I/O ----------
def utc_now():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def json_safe(o):
    """Strict-JSON friendly copy: NaN/inf/NaT/NA -> None, numpy scalars -> Python, dates -> ISO strings."""
    if o is None or o is pd.NaT or o is pd.NA:
        return None
    if isinstance(o, dict):
        return {str(k): json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [json_safe(v) for v in o]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
        return o.isoformat()
    if isinstance(o, Path):
        return str(o)
    return o


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(json_safe(obj), indent=2, ensure_ascii=False, allow_nan=False), 'utf-8')
    os.replace(tmp, path)


def flatten_records(records):
    """List of dicts -> DataFrame; nested dict/list values become JSON strings (strict)."""
    rows = [{k: (json.dumps(json_safe(v), ensure_ascii=False, allow_nan=False) if isinstance(v, (dict, list, tuple)) else v)
             for k, v in r.items()} for r in records]
    return pd.DataFrame(rows)


def write_csv(path, df):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df = df.copy()
    for c in df.columns:
        if df[c].map(lambda v: isinstance(v, (dict, list, tuple))).any():
            df[c] = df[c].map(lambda v: json.dumps(json_safe(v), ensure_ascii=False) if isinstance(v, (dict, list, tuple)) else v)
    df.to_csv(path, index=False)
    return path


def write_table(base, df):
    """Parquet when pyarrow is installed, else gzip CSV. Returns the written path (str)."""
    base = Path(base)
    base.parent.mkdir(parents=True, exist_ok=True)
    try:
        import pyarrow  # noqa: F401
        p = base.with_name(base.name + '.parquet')
        df.to_parquet(p, index=False)
    except ImportError:
        p = base.with_name(base.name + '.csv.gz')
        df.to_csv(p, index=False)
    return str(p)


def functions_cell_source(nb_path):
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    srcs = [''.join(c['source']) if isinstance(c['source'], list) else c['source'] for c in nb['cells'] if c['cell_type'] == 'code']
    hits = [s for s in srcs if s.lstrip().startswith('# FUNCTIONS')]
    if len(hits) != 1:
        raise ValueError(f'{nb_path}: expected exactly one FUNCTIONS cell, found {len(hits)}')
    return hits[0]


def verify_v1_snapshot(v1_root, manifest):
    """sha256 of every v1 raw snapshot file; raises when any differs from the v1 source_manifest (read-only check)."""
    out, bad = {}, []
    for f in manifest['files']:
        p = Path(v1_root) / f['path']
        sha = sha256_file(p) if p.is_file() else None
        if sha != f['sha256']:
            bad.append(f"{f['path']}: {sha} != {f['sha256']}")
        out[f['path']] = sha
    if bad:
        raise RuntimeError('v1 raw snapshot does not match its manifest (v1 must stay unchanged):\n' + '\n'.join(bad))
    return out


# ---------- network: allowlist, bounded retries, size cap, 429 handling ----------
class FetchError(Exception):
    def __init__(self, msg, code=None, retry_after=None):
        super().__init__(msg)
        self.code = code
        self.retry_after = retry_after


class RateLimited(FetchError):
    pass


class BudgetExhausted(FetchError):
    pass


class CacheIntegrityError(Exception):
    pass


class RequestBudget:
    """Counts network attempts (including retries); a child budget also charges its parent."""
    def __init__(self, max_requests, parent=None):
        self.max_requests = int(max_requests)
        self.used = 0
        self.parent = parent

    def take(self):
        if self.used >= self.max_requests:
            raise BudgetExhausted(f'network request budget {self.max_requests} exhausted')
        if self.parent is not None:
            self.parent.take()
        self.used += 1


def host_allowed(url, allowed_hosts):
    u = urlparse(str(url))
    return u.scheme == 'https' and (u.hostname or '').lower() in {h.lower() for h in allowed_hosts}


def make_opener(allowed_hosts):
    class _AllowlistRedirect(HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            if not host_allowed(newurl, allowed_hosts):
                raise FetchError(f'redirect to non-allowlisted URL refused: {newurl}')
            return super().redirect_request(req, fp, code, msg, headers, newurl)
    op = build_opener(_AllowlistRedirect)
    return lambda req, timeout: op.open(req, timeout=timeout)


def parse_retry_after(value):
    if value is None:
        return None
    try:
        return max(0.0, float(str(value).strip()))
    except ValueError:
        return None  # HTTP-date form is treated as unknown -> stop rather than guess


def http_get(url, net, budget, opener=None, sleep=time.sleep, max_bytes=None, log=print):
    """GET with https+allowlist (also after redirects), bounded retries/backoff, response-size cap and 429 Retry-After."""
    if not host_allowed(url, net['allowed_hosts']):
        raise FetchError(f'URL not allowlisted (https and an allowed host required): {url}')
    opener = opener or make_opener(net['allowed_hosts'])
    cap = int(max_bytes or net['max_response_bytes'])
    retries = int(net['max_retries'])
    attempt = 0
    while True:
        budget.take()
        attempt += 1
        try:
            req = Request(url, headers={'User-Agent': net['user_agent'], 'Accept': '*/*'})
            with opener(req, float(net['timeout_seconds'])) as resp:
                final = resp.geturl()
                if not host_allowed(final, net['allowed_hosts']):
                    raise FetchError(f'resolved URL not allowlisted: {final}')
                body = resp.read(cap + 1)
                if len(body) > cap:
                    raise FetchError(f'response exceeds max_response_bytes={cap}: {url}')
                h = resp.headers
                return {'url': url, 'resolved_url': final, 'http_status': getattr(resp, 'status', None), 'attempts': attempt,
                        'retrieved_at_utc': utc_now(), 'body': body,
                        'headers': {k: h.get(k) for k in ('Content-Type', 'Content-Length', 'Last-Modified', 'ETag')}}
        except HTTPError as e:
            ra = parse_retry_after(e.headers.get('Retry-After') if e.headers is not None else None)
            try:
                snippet = e.read(500).decode('utf-8', errors='replace')
            except Exception:
                snippet = ''
            if e.code == 429:
                if ra is not None and ra <= float(net['max_retry_after_seconds']) and attempt <= retries:
                    log(f'    HTTP 429 for {url}; waiting Retry-After={ra}s')
                    sleep(ra)
                    continue
                raise RateLimited(f'HTTP 429 rate limited (Retry-After={ra}) for {url}; stopping instead of retrying. {snippet}',
                                  code=429, retry_after=ra)
            if e.code in (500, 502, 503, 504) and attempt <= retries:
                sleep(float(net['backoff_seconds']) * 2 ** (attempt - 1))
                continue
            raise FetchError(f'HTTP {e.code} for {url}. {snippet}'.strip(), code=e.code)
        except FetchError:
            raise
        except (URLError, socket.timeout, TimeoutError, ConnectionError, OSError) as e:
            if attempt <= retries:
                sleep(float(net['backoff_seconds']) * 2 ** (attempt - 1))
                continue
            raise FetchError(f'network error for {url} after {attempt} attempts: {type(e).__name__}: {e}')


# ---------- hash-verified cache registry ----------
def load_registry(path):
    p = Path(path)
    if p.is_file():
        return json.loads(p.read_text('utf-8'))
    return {'schema': 'roadaware-external-v2-registry/1', 'resources': {}}


def _write_bytes(path, body):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.part')
    tmp.write_bytes(body)
    os.replace(tmp, path)


def _free_version_path(root, rel, start):
    p = Path(rel)
    n = start
    while (Path(root) / p.with_name(f'{p.stem}.v{n}{p.suffix}')).exists():
        n += 1
    return p.with_name(f'{p.stem}.v{n}{p.suffix}').as_posix()


def cached_fetch(res_id, url, rel, root, registry, net, budget, opener=None, sleep=time.sleep, refresh=False, offline=False,
                 validate=None, meta=None, max_bytes=None, log=print):
    """Return the registry entry for res_id. A cached file is reused only when its sha256 matches the registry; a mismatch
    raises (file preserved, never overwritten). Re-downloaded bytes that differ from the registry become a new version file."""
    root = Path(root)
    res = registry.setdefault('resources', {})
    entry = res.get(res_id)
    has_reg = bool(entry and entry.get('sha256') and entry.get('path'))
    if has_reg and (root / entry['path']).is_file():
        sha = sha256_file(root / entry['path'])
        if sha != entry['sha256']:
            raise CacheIntegrityError(f"{res_id}: cached file {entry['path']} sha256 {sha[:12]} != registry {entry['sha256'][:12]}; "
                                      'file preserved (not overwritten). Restore it or move it aside, then rerun.')
        if not refresh:
            entry.update(last_status='cache_hit_hash_verified', verified_at_utc=utc_now())
            return entry
    elif not has_reg and (root / rel).is_file():
        raise CacheIntegrityError(f'{res_id}: unregistered file already at {rel}; refusing to adopt or overwrite it. Move it aside and rerun.')
    if offline:
        raise FetchError(f'{res_id}: offline mode and no hash-verified cached copy')
    r = http_get(url, net, budget, opener=opener, sleep=sleep, max_bytes=max_bytes, log=log)
    body = r.pop('body')
    if validate is not None:
        validate(body)
    sha = sha256_bytes(body)
    versions = list(entry.get('versions', [])) if entry else []
    if has_reg and sha == entry['sha256']:
        path, status = entry['path'], 'refetched_identical'
    elif has_reg:
        versions.append({k: entry.get(k) for k in ('path', 'sha256', 'bytes', 'retrieved_at_utc', 'verified_at_utc')})
        path, status = _free_version_path(root, entry['path'], len(versions) + 1), 'remote_bytes_changed_new_version'
    else:
        path, status = Path(rel).as_posix(), 'downloaded'
    if not (root / path).is_file():
        _write_bytes(root / path, body)
    new = {'resource_id': res_id, 'url': url, 'resolved_url': r['resolved_url'], 'path': path, 'sha256': sha, 'bytes': len(body),
           'retrieved_at_utc': r['retrieved_at_utc'], 'verified_at_utc': utc_now(), 'http_status': r['http_status'],
           'attempts': r['attempts'], 'headers': r['headers'], 'provenance': 'downloaded_by_notebook09', 'last_status': status,
           'versions': versions}
    new.update(meta or {})
    res[res_id] = new
    return new


def try_fetch(res_id, url, rel, root, registry, net, budget, required=False, **kw):
    """cached_fetch, but optional resources that fail are recorded as unavailable (never faked). Integrity errors always raise."""
    try:
        return cached_fetch(res_id, url, rel, root, registry, net, budget, **kw)
    except FetchError as e:
        if required:
            raise
        res = registry.setdefault('resources', {})
        e0 = dict(res.get(res_id) or {})
        e0.update(resource_id=res_id, url=url, last_status='unavailable', error=str(e)[:1000], error_type=type(e).__name__,
                  attempted_at_utc=utc_now())
        if not e0.get('sha256'):
            e0['path'] = None
        res[res_id] = e0
        print(f'    unavailable: {res_id}: {str(e)[:300]}')
        out = dict(e0)
        out['sha256'] = None  # this run has no usable bytes
        return out


def register_preflight(res_id, url, path, registry, provenance, meta=None):
    """Register a Codex preflight file (read-only): hash, size and cache mtime; original download time stays unknown.
    If the bytes differ from an earlier registration the old record is kept under versions and the change is flagged."""
    p = Path(path)
    sha = sha256_file(p)
    st = p.stat()
    res = registry.setdefault('resources', {})
    old = res.get(res_id)
    versions = list(old.get('versions', [])) if old else []
    status = 'preflight_hash_registered'
    if old and old.get('sha256'):
        if old['sha256'] == sha:
            status = 'preflight_hash_verified'
        else:
            versions.append({k: old.get(k) for k in ('sha256', 'bytes', 'cache_mtime_utc', 'verified_at_utc')})
            status = 'preflight_changed_since_registry_new_version'
    e = {'resource_id': res_id, 'url': url, 'preflight_path': str(p), 'path': None, 'sha256': sha, 'bytes': st.st_size,
         'cache_mtime_utc': dt.datetime.fromtimestamp(st.st_mtime, dt.timezone.utc).replace(microsecond=0).isoformat(),
         'verified_at_utc': utc_now(), 'retrieved_at_utc': None, 'original_download_time': 'unknown', 'provenance': provenance,
         'last_status': status, 'versions': versions}
    e.update(meta or {})
    res[res_id] = e
    return e


def json_validator(require_success=False):
    def v(body):
        try:
            o = json.loads(body.decode('utf-8'))
        except Exception as e:
            raise FetchError(f'invalid JSON response: {e}')
        if require_success and not (isinstance(o, dict) and o.get('success') is True):
            raise FetchError('CKAN response without success == true')
    return v


def nonempty_validator(body):
    if not body or not body.strip():
        raise FetchError('empty response body')


def xlsx_validator(body):
    if not zipfile.is_zipfile(io.BytesIO(body)):
        raise FetchError('response is not a complete XLSX (zip) file')


def pdf_validator(body):
    if not body.lstrip()[:5].startswith(b'%PDF'):
        raise FetchError('response is not a PDF')


def weather_validator(n_locations):
    def v(body):
        try:
            o = json.loads(body.decode('utf-8'))
        except Exception as e:
            raise FetchError(f'invalid weather JSON: {e}')
        if isinstance(o, dict) and o.get('error'):
            raise FetchError(f"Open-Meteo error: {o.get('reason')}")
        items = o if isinstance(o, list) else [o]
        if len(items) != n_locations or not all(isinstance(i, dict) and 'hourly' in i for i in items):
            raise FetchError(f'weather response has {len(items)} location blocks, expected {n_locations} with hourly data')
    return v


# ---------- date evidence ----------
def norm_id(v):
    """Exact-ID normalisation only: whitespace collapse, integral floats ('123.0', 123.0) -> '123'. No fuzzy matching."""
    if v is None:
        return ''
    if isinstance(v, (bool, np.bool_)):
        return str(v)
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        if not math.isfinite(float(v)):
            return ''
        return str(int(v)) if float(v).is_integer() else repr(float(v))
    s = re.sub(r'\s+', ' ', str(v)).strip()
    m = re.fullmatch(r'(\d+)\.0+', s)
    return m.group(1) if m else s


def csv_date_candidates(raw, year):
    """Readings of one original CSV date string: dict(kind, mdy, dmy, candidates in `year`). Never chooses between readings."""
    s = re.sub(r'\s+', ' ', '' if raw is None else str(raw)).strip()
    out = {'kind': 'missing', 'mdy': None, 'dmy': None, 'candidates': ()}
    if not s:
        return out
    if re.fullmatch(r'\d+(\.0+)?', s):
        n = int(float(s))
        if 30000 <= n <= 60000:
            d = EXCEL_ORIGIN + dt.timedelta(days=n)
            out.update(kind='excel_serial', candidates=(d,) if d.year == year else ())
        else:
            out['kind'] = 'invalid_number'
        return out
    m = re.fullmatch(r'(\d{4})-(\d{1,2})-(\d{1,2})', s)
    if m:
        try:
            d = dt.date(*map(int, m.groups()))
            out.update(kind='iso', candidates=(d,) if d.year == year else ())
        except ValueError:
            out['kind'] = 'invalid_iso'
        return out
    m = re.fullmatch(r'(\d{1,2})/(\d{1,2})/(\d{4})', s)
    if m:
        a, b, y = map(int, m.groups())
        y = y - 543 if y > 2400 else y

        def mk(mm, dd):
            try:
                return dt.date(y, mm, dd)
            except ValueError:
                return None
        mdy, dmy = mk(a, b), mk(b, a)
        cands = tuple(sorted({d for d in (mdy, dmy) if d is not None and d.year == year}))
        out.update(kind='slash', mdy=mdy, dmy=dmy, candidates=cands)
        return out
    out['kind'] = 'unrecognized'
    return out


def date_format_profile(series, year):
    """Per-file format profile of raw date strings (evidence is yearly, never one global rule)."""
    kinds, shapes = {}, {'first_gt12': 0, 'second_gt12': 0, 'equal_parts': 0, 'both_le12_ambiguous': 0}
    for v, n in series.astype(str).value_counts().items():
        c = csv_date_candidates(v, year)
        kinds[c['kind']] = kinds.get(c['kind'], 0) + int(n)
        m = re.fullmatch(r'\s*(\d{1,2})/(\d{1,2})/\d{4}\s*', v)
        if m:
            a, b = int(m.group(1)), int(m.group(2))
            key = 'equal_parts' if a == b else ('first_gt12' if a > 12 else ('second_gt12' if b > 12 else 'both_le12_ambiguous'))
            shapes[key] += int(n)
    return {'csv_format_summary': ';'.join(f'{k}:{v}' for k, v in sorted(kinds.items())),
            'csv_excel_serial_rows': kinds.get('excel_serial', 0), 'csv_slash_rows': kinds.get('slash', 0),
            'csv_ambiguous_slash_rows': shapes['both_le12_ambiguous'], 'csv_slash_first_part_gt12_rows': shapes['first_gt12'],
            'csv_slash_second_part_gt12_rows': shapes['second_gt12'], 'csv_slash_equal_parts_rows': shapes['equal_parts']}


def classify_xlsx_date(id_value, date_value, xlsx_row):
    if isinstance(date_value, dt.datetime):
        kind, d = 'typed_datetime', date_value.date().isoformat()
    elif isinstance(date_value, dt.date):
        kind, d = 'typed_date', date_value.isoformat()
    elif date_value is None or (isinstance(date_value, str) and not date_value.strip()):
        kind, d = 'missing', None
    elif isinstance(date_value, (int, float)) and not isinstance(date_value, bool):
        kind, d = 'untyped_numeric', None  # a plain number is not typed date evidence
    else:
        kind, d = 'text', None
    return {'acc_code': norm_id(id_value), 'xlsx_date': d, 'xlsx_kind': kind, 'xlsx_row': xlsx_row, 'xlsx_raw': str(date_value)[:40]}


def read_xlsx_id_dates(path, id_header='ACC_CODE', date_header='วันที่เกิดเหตุ', max_header_scan=30):
    """(DataFrame acc_code/xlsx_date/xlsx_kind/xlsx_row, meta) from the first sheet whose header row has both columns."""
    from openpyxl import load_workbook
    wb = load_workbook(path, read_only=True, data_only=True)
    try:
        for ws in wb.worksheets:
            it = ws.iter_rows(values_only=True)
            for i, row in enumerate(it):
                if i >= max_header_scan:
                    break
                names = [re.sub(r'\s+', '', str(c)) if c is not None else '' for c in (row or ())]
                if id_header in names and date_header in names:
                    ci, cd = names.index(id_header), names.index(date_header)
                    recs, blank = [], 0
                    for j, r in enumerate(it):
                        if r is None or all(c is None or (isinstance(c, str) and not c.strip()) for c in r):
                            blank += 1
                            continue
                        recs.append(classify_xlsx_date(r[ci] if ci < len(r) else None, r[cd] if cd < len(r) else None, i + j + 2))
                    df = pd.DataFrame(recs, columns=['acc_code', 'xlsx_date', 'xlsx_kind', 'xlsx_row', 'xlsx_raw'])
                    return df, {'sheet': ws.title, 'sheets': list(wb.sheetnames), 'header_row': i + 1, 'blank_rows_skipped': blank,
                                'rows': len(df), 'kind_counts': df['xlsx_kind'].value_counts().to_dict()}
        raise ValueError(f'{path}: no header row with {id_header!r} and {date_header!r} in the first {max_header_scan} rows of any sheet')
    finally:
        wb.close()


def audit_dates_by_id(csv_rows, xlsx_rows, year):
    """csv_rows: source_row, acc_code, date_raw (original v1 CSV). xlsx_rows: acc_code, xlsx_date (ISO or None), xlsx_kind.
    Exact ID join only. A CSV row is 'resolved' only when every XLSX row of its ID carries the same TYPED date, every CSV row
    of the ID has the same raw string, and that date is one of the row's own candidate readings. Returns (rows, summary)."""
    c = csv_rows[['source_row', 'acc_code', 'date_raw']].copy()
    c['acc_code'] = c['acc_code'].map(norm_id)
    parsed = {v: csv_date_candidates(v, year) for v in c['date_raw'].unique()}
    x = xlsx_rows.copy()
    x['acc_code'] = x['acc_code'].map(norm_id)
    x = x[x['acc_code'] != '']
    xg = {}
    for k, g in x.groupby('acc_code', sort=False):
        typed = g[g['xlsx_kind'].isin(TYPED_KINDS)]
        xg[k] = (len(g), set(typed['xlsx_date']), len(typed) < len(g))
    with_id = c[c['acc_code'] != '']
    csv_n = with_id.groupby('acc_code').size().to_dict()
    csv_raw_n = with_id.groupby('acc_code')['date_raw'].nunique().to_dict()
    out = {'csv_kind': [], 'ambiguous_in_v1': [], 'resolution_status': [], 'xlsx_reading': [], 'resolved_date': [],
           'xlsx_rows_for_id': [], 'csv_rows_for_id': []}
    for a, raw in zip(c['acc_code'], c['date_raw']):
        p = parsed[raw]
        st, rd, rv, nx = None, None, None, 0
        if a == '':
            st = 'no_id'
        elif a not in xg:
            st = 'id_absent_in_xlsx'
        else:
            nx, dates, untyped = xg[a]
            if untyped:
                st = 'xlsx_untyped_or_missing_date'
            elif len(dates) != 1:
                st = 'xlsx_id_conflicting_dates'
            elif csv_raw_n[a] > 1:
                st = 'csv_id_conflicting_dates'
            elif not p['candidates']:
                st = 'csv_date_unparseable'
            else:
                xd = dt.date.fromisoformat(next(iter(dates)))
                if xd not in p['candidates']:
                    st = 'mismatch'
                else:
                    st, rv = 'resolved', xd.isoformat()
                    if p['kind'] != 'slash':
                        rd = 'non_slash'
                    elif p['mdy'] == p['dmy']:
                        rd = 'equal_parts'
                    else:
                        rd = 'mdy' if xd == p['mdy'] else 'dmy'
        out['csv_kind'].append(p['kind'])
        out['ambiguous_in_v1'].append(len(p['candidates']) > 1)
        out['resolution_status'].append(st)
        out['xlsx_reading'].append(rd)
        out['resolved_date'].append(rv)
        out['xlsx_rows_for_id'].append(nx)
        out['csv_rows_for_id'].append(csv_n.get(a, 0) if a else 0)
    for k, v in out.items():
        c[k] = v
    sc = c['resolution_status'].value_counts().to_dict()
    rc = c.loc[c['resolution_status'] == 'resolved', 'xlsx_reading'].value_counts().to_dict()
    n_with_id = int((c['acc_code'] != '').sum())
    n_mdy, n_dmy, n_mis = rc.get('mdy', 0), rc.get('dmy', 0), sc.get('mismatch', 0)
    if n_mdy and not n_dmy and not n_mis and sc.get('resolved', 0) == n_with_id:
        concl = 'mdy_all_id_rows_agree'
    elif n_mdy and not n_dmy:
        concl = 'mdy_only_among_resolved_some_rows_unresolved'
    elif n_dmy and not n_mdy:
        concl = 'dmy_only_among_resolved' + ('' if sc.get('resolved', 0) == n_with_id else '_some_rows_unresolved')
    elif n_mdy and n_dmy:
        concl = 'mixed_readings_per_id_resolution_only'
    else:
        concl = 'no_distinguishing_evidence'
    csv_ids, x_ids = set(csv_n), set(xg)
    amb = c['ambiguous_in_v1']
    summary = {
        'n_csv_rows': len(c), 'n_csv_rows_with_id': n_with_id, 'n_csv_unique_ids': len(csv_ids),
        'n_csv_duplicate_id_groups': sum(1 for v in csv_n.values() if v > 1),
        'n_csv_ids_conflicting_dates': sum(1 for v in csv_raw_n.values() if v > 1),
        'n_xlsx_rows': len(xlsx_rows), 'n_xlsx_rows_with_id': len(x), 'n_xlsx_unique_ids': len(x_ids),
        'n_xlsx_duplicate_id_groups': sum(1 for v in xg.values() if v[0] > 1),
        'xlsx_kind_counts': xlsx_rows['xlsx_kind'].value_counts().to_dict(),
        'n_ids_csv_only': len(csv_ids - x_ids), 'n_ids_xlsx_only_not_added': len(x_ids - csv_ids),
        'n_ids_row_count_differs': sum(1 for k in csv_ids & x_ids if csv_n[k] != xg[k][0]),
        'status_counts': sc, 'reading_counts': rc, 'n_resolved': sc.get('resolved', 0), 'n_mismatch': n_mis,
        'n_ambiguous_in_v1': int(amb.sum()), 'n_ambiguous_resolved': int((amb & (c['resolution_status'] == 'resolved')).sum()),
        'n_ambiguous_unresolved': int((amb & (c['resolution_status'] != 'resolved')).sum()), 'conclusion': concl,
    }
    return c, summary


# ---------- traffic (DOH AADT) ----------
def resource_meta(res):
    res = res or {}
    return {'resource_id': res.get('id'), 'resource_name': res.get('name'), 'format': res.get('format'), 'url': res.get('url'),
            'catalog_created': res.get('created'), 'catalog_last_modified': res.get('last_modified'),
            'catalog_metadata_modified': res.get('metadata_modified'),
            'catalog_data_release': res.get('data_release_date') or res.get('resource_data_release_calendar'),
            'catalog_resource_last_updated': res.get('resource_last_updated_date'), 'catalog_size': res.get('size'),
            'catalog_hash': res.get('hash') or None, 'catalog_unit': res.get('resource_unit_of_measure'),
            'available_from_proxy': res.get('created'),
            'available_from_note': 'catalog created timestamp is a publication proxy, not proof the current bytes existed then'}


def resource_year_be(res):
    m = re.search(r'(25\d\d)', str(res.get('name') or ''))
    if m:
        return int(m.group(1))
    m = re.search(r'aadt[-_](\d\d)[-_.]', str(res.get('url') or '').lower())
    return 2500 + int(m.group(1)) if m else None


def select_aadt_resources(resources, years_be, prefer_format='CSV'):
    """{be_year: (chosen resource, [alternatives])}; latest 'created' wins among same-year resources of the preferred format."""
    out = {}
    for be in years_be:
        cands = [r for r in resources if str(r.get('format', '')).upper() == prefer_format.upper() and resource_year_be(r) == be]
        if cands:
            cands = sorted(cands, key=lambda r: str(r.get('created') or ''), reverse=True)
            out[be] = (cands[0], cands[1:])
    return out


def read_csv_bytes(body, encodings):
    for enc in encodings:
        try:
            return pd.read_csv(io.BytesIO(body), dtype=str, keep_default_na=False, encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f'CSV not decodable with {encodings}')


def norm_header(h):
    return re.sub(r'\s+', '', str('' if h is None else h)).replace('﻿', '')


def to_number(s):
    """'1,234' -> 1234.0; '', '-', text -> NaN (never 0)."""
    t = s.astype(str).str.replace(',', '', regex=False).str.strip()
    return pd.to_numeric(t.where(~t.isin(['', '-', 'nan', 'None', 'NaN'])), errors='coerce')


def normalize_aadt(raw, aliases, required, total_components, heavy_components, be_year):
    hdr = {}
    for c in raw.columns:
        hdr.setdefault(norm_header(c), c)
    mapped = {}
    for canon, names in aliases.items():
        hit = [hdr[norm_header(n)] for n in names if norm_header(n) in hdr]
        if hit:
            mapped[canon] = hit[0]
    missing = [k for k in required if k not in mapped]
    out = pd.DataFrame({'survey_year_be': be_year, 'survey_year_ce': be_year - 543}, index=raw.index)
    for canon in ('province', 'route', 'control_section', 'survey_point', 'route_name', 'highway_district'):
        out[canon] = raw[mapped[canon]].astype(str).str.strip() if canon in mapped else ''
    out['total_aadt'] = to_number(raw[mapped['total']]) if 'total' in mapped else np.nan
    out['heavy_share_pct'] = to_number(raw[mapped['heavy_share_pct']]) if 'heavy_share_pct' in mapped else np.nan
    used = set(mapped.values())
    nonnumeric, unmapped_text = {}, []
    for c in raw.columns:
        if c in used:
            continue
        v = to_number(raw[c])
        nonempty = raw[c].astype(str).str.strip().ne('')
        if nonempty.any() and v.notna().sum() >= 0.5 * nonempty.sum():
            out['count__' + norm_header(c)] = v
            nonnumeric[norm_header(c)] = int((nonempty & v.isna()).sum())
        else:
            unmapped_text.append(norm_header(c))
    for c in ('total_aadt', 'heavy_share_pct'):
        src = mapped.get('total' if c == 'total_aadt' else 'heavy_share_pct')
        if src is not None:
            ne = raw[src].astype(str).str.strip().ne('')
            nonnumeric[c] = int((ne & out[c].isna()).sum())
    comp = ['count__' + norm_header(n) for n in total_components]
    if all(c in out for c in comp) and 'total' in mapped:
        diff = (out[comp].sum(axis=1, min_count=len(comp)) - out['total_aadt']).abs()
        total_check = {'evaluated': True, 'n_compared': int(diff.notna().sum()), 'n_mismatch_gt1': int((diff > 1).sum())}
    else:
        total_check = {'evaluated': False, 'missing_components': [c for c in comp if c not in out], 'n_mismatch_gt1': None}
    hv = ['count__' + norm_header(n) for n in heavy_components]
    if all(c in out for c in hv) and 'total' in mapped and 'heavy_share_pct' in mapped:
        share = 100.0 * out[hv].sum(axis=1, min_count=len(hv)) / out['total_aadt'].where(out['total_aadt'] > 0)
        d = (share - out['heavy_share_pct']).abs()
        heavy_check = {'evaluated': True, 'n_compared': int(d.notna().sum()), 'n_mismatch_gt0_05pp': int((d > 0.05).sum())}
    else:
        heavy_check = {'evaluated': False, 'n_mismatch_gt0_05pp': None}
    key = ['route', 'control_section', 'survey_point']
    dup = int(out.duplicated(key, keep=False).sum()) if all(k in mapped for k in key) else None
    num_cols = [c for c in out.columns if c.startswith('count__')] + ['total_aadt']
    neg = int((out[num_cols] < 0).sum().sum())
    rep = {'status': 'normalized' if not missing else 'schema_incomplete', 'mapped': mapped, 'missing_fields': missing,
           'unmapped_text_headers': unmapped_text, 'duplicate_key_rows': dup, 'total_check': total_check, 'heavy_check': heavy_check,
           'negative_values': neg, 'nonnumeric_cells': nonnumeric, 'unit': 'vehicles/day (AADT) at a survey point'}
    return out, rep


# ---------- geography ----------
def norm_en(s):
    return re.sub(r'[^a-z0-9]', '', str(s or '').lower())


def _pick_key(keys, candidates):
    low = {str(k).lower(): k for k in keys}
    for c in candidates:
        if c.lower() in low:
            return low[c.lower()]
    return None


def province_table(obj):
    rows = obj
    if isinstance(obj, dict):
        rows = next((obj[k] for k in ('data', 'provinces', 'items') if isinstance(obj.get(k), list)), None)
    if not isinstance(rows, list) or not rows or not isinstance(rows[0], dict):
        raise ValueError('province table: expected a JSON list of objects')
    keys = list(rows[0].keys())
    kc = _pick_key(keys, ['provinceCode', 'province_code', 'code', 'PROVINCE_CODE'])
    kt = _pick_key(keys, ['provinceNameTh', 'province_name_th', 'nameTh', 'name_th'])
    ke = _pick_key(keys, ['provinceNameEn', 'province_name_en', 'nameEn', 'name_en'])
    if not (kc and kt and ke):
        raise ValueError(f'province table keys not recognised (need code/Thai/English name): {sorted(keys)}')
    df = pd.DataFrame({'province_code': [int(r[kc]) for r in rows], 'name_th': [str(r[kt]).strip() for r in rows],
                       'name_en': [str(r[ke]).strip() for r in rows]})
    if df['province_code'].duplicated().any():
        raise ValueError('province table has duplicate province codes')
    return df, {'code_key': kc, 'name_th_key': kt, 'name_en_key': ke}


def join_shapes(shapes, provinces):
    """ISO 3166-2 'TH-NN' -> province code first, else EXACT normalized English name; never fuzzy. Unresolved stay explicit."""
    codes = set(provinces['province_code'])
    by_name = {}
    for code, en in zip(provinces['province_code'], provinces['name_en']):
        by_name.setdefault(norm_en(en), []).append(code)
    recs = []
    for idx, name, iso in zip(shapes['shape_index'], shapes['shape_name'], shapes['shape_iso']):
        m = re.fullmatch(r'TH-(\d{2})', str(iso or '').strip())
        code, how = None, 'unresolved_no_iso_or_exact_name'
        if m and int(m.group(1)) in codes:
            code, how = int(m.group(1)), 'iso_3166_2_code'
        else:
            hits = by_name.get(norm_en(name), [])
            if len(hits) == 1:
                code, how = hits[0], 'exact_normalized_english_name'
            elif len(hits) > 1:
                how = 'unresolved_name_not_unique'
        recs.append({'shape_index': idx, 'province_code': code, 'join_method': how})
    j = pd.DataFrame(recs, columns=['shape_index', 'province_code', 'join_method'])
    dup = j['province_code'].notna() & j['province_code'].duplicated(keep=False)
    j.loc[dup, 'join_method'] = 'unresolved_duplicate_code'
    j.loc[dup, 'province_code'] = None
    j['province_code'] = pd.array(j['province_code'], dtype='Int64')
    return j


def representative_points(features):
    from shapely.geometry import shape
    recs = []
    for i, f in enumerate(features):
        props = f.get('properties') or {}
        rec = {'shape_index': i, 'shape_name': props.get('shapeName'), 'shape_iso': props.get('shapeISO'), 'shape_id': props.get('shapeID'),
               'lat': None, 'lon': None, 'point_within_polygon': False, 'geometry_valid': False}
        if f.get('geometry'):
            g = shape(f['geometry'])
            rec['geometry_valid'] = bool(g.is_valid)
            if not g.is_empty:
                p = g.representative_point()
                rec.update(lat=float(p.y), lon=float(p.x), point_within_polygon=bool(g.contains(p)))
        recs.append(rec)
    return pd.DataFrame(recs)


# ---------- weather (archived forecasts) ----------
def estimate_call_weight(n_locations, n_days, n_vars):
    """Approximate Open-Meteo fair-use weight (locations x 2-week units x variable factor); an estimate, not the official count."""
    return round(n_locations * max(1.0, n_days / 14.0) * max(1.0, n_vars / 10.0), 1)


def weather_url(base, batch, start, end, w):
    q = {'latitude': ','.join(f'{v:.4f}' for v in batch['lat']), 'longitude': ','.join(f'{v:.4f}' for v in batch['lon']),
         'hourly': ','.join(w['hourly_variables']), 'models': w['model'], 'timezone': w['timezone'], 'start_date': start, 'end_date': end}
    return base + '?' + urlencode(q, safe=',')


def normalize_weather_payload(payload, batch, variables, model, timezone, utc_offset_seconds, lead_hours):
    """Hourly rows per location with valid time (local + UTC) and the conservative nominal issue bound = valid UTC - lead."""
    items = payload if isinstance(payload, list) else [payload]
    if len(items) != len(batch):
        raise ValueError(f'{len(items)} location blocks for {len(batch)} requested points')
    frames = []
    for it, pt in zip(items, batch.itertuples(index=False)):
        if it.get('error'):
            raise ValueError(f"Open-Meteo error: {it.get('reason')}")
        if it.get('utc_offset_seconds') != utc_offset_seconds or it.get('timezone') != timezone:
            raise ValueError(f"unexpected timezone {it.get('timezone')}/{it.get('utc_offset_seconds')} (expected {timezone}/{utc_offset_seconds})")
        h = it['hourly']
        t = pd.to_datetime(pd.Series(h['time']), format='%Y-%m-%dT%H:%M')
        f = pd.DataFrame({'province_code': int(pt.province_code), 'valid_time_local': t})
        for v in variables:
            key = v if v in h else next((k for k in h if k.startswith(v)), None)
            if key is None:
                raise ValueError(f'variable {v} missing from response')
            vals = pd.to_numeric(pd.Series(h[key], dtype=object), errors='coerce')
            if len(vals) != len(t):
                raise ValueError(f'{v}: {len(vals)} values for {len(t)} timestamps')
            f[v] = vals.to_numpy(dtype=float)
        f['valid_time_utc'] = f['valid_time_local'] - pd.Timedelta(seconds=utc_offset_seconds)
        f['nominal_issue_bound_utc'] = f['valid_time_utc'] - pd.Timedelta(hours=lead_hours)
        f['requested_lat'], f['requested_lon'] = float(pt.lat), float(pt.lon)
        f['grid_lat'], f['grid_lon'], f['grid_elevation'] = it.get('latitude'), it.get('longitude'), it.get('elevation')
        f['timezone'], f['utc_offset_seconds'], f['model'], f['lead_hours'] = it.get('timezone'), it.get('utc_offset_seconds'), model, lead_hours
        frames.append(f)
    return pd.concat(frames, ignore_index=True)


def aggregate_weather(hourly, block_hours, temp_col, rain_col, buffer_hours, issue_hour_local, utc_offset_seconds):
    """Per province x local date x block (24 = daily, 6 = 00-06/06-12/12-18/18-24). Sums/means only from COMPLETE hours
    (all-null is never 0). issue_bound_ok: latest nominal issue bound + buffer <= (target date - 1) at issue_hour_local."""
    h = hourly.copy()
    h['date'] = h['valid_time_local'].dt.normalize()
    h['block'] = h['valid_time_local'].dt.hour // block_hours
    g = h.groupby(['province_code', 'date', 'block'])
    a = g.agg(n_rows=('valid_time_local', 'size'), n_hours=('valid_time_local', 'nunique'), n_valid_rain=(rain_col, 'count'),
              n_valid_temp=(temp_col, 'count'), _rain=(rain_col, 'sum'), _temp=(temp_col, 'mean'),
              earliest_issue_bound_utc=('nominal_issue_bound_utc', 'min'), latest_issue_bound_utc=('nominal_issue_bound_utc', 'max')).reset_index()
    a['n_expected'] = block_hours
    clean = (a['n_rows'] == block_hours) & (a['n_hours'] == block_hours)
    a['rain_missing'] = ~(clean & (a['n_valid_rain'] == block_hours))
    a['temp_missing'] = ~(clean & (a['n_valid_temp'] == block_hours))
    a['rain_sum'] = a['_rain'].where(~a['rain_missing'])
    a['temp_mean'] = a['_temp'].where(~a['temp_missing'])
    a = a.drop(columns=['_rain', '_temp'])
    a['time_block'] = 'day' if block_hours == 24 else a['block'].map(lambda b: f'{b * block_hours:02d}-{(b + 1) * block_hours:02d}')
    issue_local = a['date'] - pd.Timedelta(days=1) + pd.Timedelta(hours=issue_hour_local)
    a['issue_time_utc'] = issue_local - pd.Timedelta(seconds=utc_offset_seconds)
    a['issue_bound_ok'] = (a['latest_issue_bound_utc'] + pd.Timedelta(hours=buffer_hours)) <= a['issue_time_utc']
    return a


# ---------- calendar ----------
def html_to_text(html):
    s = re.sub(r'(?is)<(script|style)\b.*?</\1>', ' ', html)
    s = re.sub(r'(?i)<(br|/p|/div|/li|/tr|/h\d)[^>]*>', '\n', s)
    s = re.sub(r'<[^>]+>', ' ', s)
    return '\n'.join(re.sub(r'[ \t\r\f\v]+', ' ', ln).strip() for ln in htmllib.unescape(s).splitlines() if ln.strip())


def discover_pdf_links(html, base_url, allowed_hosts):
    out = []
    for h in re.findall(r'''href\s*=\s*["']([^"']+?\.pdf)(?:[?#][^"']*)?["']''', html, flags=re.I):
        u = urljoin(base_url, htmllib.unescape(h))
        if 'holiday' in u.lower() and host_allowed(u, allowed_hosts) and u not in out:
            out.append(u)
    return out


def year_from_url(u):
    name = urlparse(u).path.rsplit('/', 1)[-1]
    m = re.search(r'(20\d\d)', name)
    if m:
        return int(m.group(1))
    m = re.search(r'(25\d\d)', name)
    return int(m.group(1)) - 543 if m else None


def pdf_text(body):
    from pypdf import PdfReader
    r = PdfReader(io.BytesIO(body))
    return '\n'.join((p.extract_text() or '') for p in r.pages), len(r.pages)


def extract_thai_dates(text, year):
    """[(date, line)] for every '<day> <Thai month>' mention (Thai or Arabic digits) in a source for calendar `year`."""
    pat = re.compile(r'(\d{1,2})\s*(' + '|'.join(THAI_MONTHS) + r')')
    out = []
    for line in str(text).translate(THAI_DIGITS).splitlines():
        for m in pat.finditer(line):
            try:
                d = dt.date(int(year), THAI_MONTHS[m.group(2)], int(m.group(1)))
            except ValueError:
                continue
            out.append({'date': d.isoformat(), 'line': re.sub(r'\s+', ' ', line).strip()[:200]})
    return out


def classify_calendar_records(found, recurring, src):
    """Only recurring fixed festivals whose keyword is on the same line are verified; everything else is explicitly excluded.
    Bank (BOT) holidays are never labelled national public holidays."""
    recs = []
    for f in found:
        d = dt.date.fromisoformat(f['date'])
        fest = next((r for r in recurring if [d.month, d.day] in [list(md) for md in r['month_days']]
                     and any(k in f['line'] for k in r['keywords'])), None)
        recs.append({'date': f['date'], 'name': fest['name'] if fest else None, 'festival_id': fest['id'] if fest else None,
                     'status': VERIFIED_FESTIVAL if fest else EXCLUDED_TEXT, 'record_type': src.get('record_type', 'bot_financial_institution_holiday'),
                     'is_national_public_holiday': None, 'source_id': src.get('source_id'), 'source_url': src.get('url'),
                     'source_sha256': src.get('sha256'), 'source_year': src.get('year'), 'announcement_date': src.get('announcement_date'),
                     'announcement_date_status': src.get('announcement_date_status', 'unknown'), 'source_text': f['line']})
    return recs


def dedupe_calendar_records(df):
    return df.drop_duplicates(['date', 'festival_id', 'status', 'source_id', 'source_text']).sort_values(['date', 'source_id']).reset_index(drop=True)


def festival_evidence(records, recurring, years):
    rows = []
    for y in years:
        for r in recurring:
            ok = records[(records['status'] == VERIFIED_FESTIVAL) & (records['festival_id'] == r['id'])
                         & (records['record_type'] == 'bot_financial_institution_holiday') & (records['source_year'] == y)]
            rows.append({'year': y, 'festival_id': r['id'], 'name': r['name'],
                         'rule_dates': ';'.join(dt.date(y, m, d).isoformat() for m, d in r['month_days']),
                         'confirmed_in_official_pdf': len(ok) > 0, 'confirmed_dates': ';'.join(sorted(set(ok['date']))),
                         'sources': ';'.join(sorted(set(ok['source_id'].astype(str))))})
    return pd.DataFrame(rows, columns=['year', 'festival_id', 'name', 'rule_dates', 'confirmed_in_official_pdf', 'confirmed_dates', 'sources'])


def calendar_feature_eligible(rec):
    """Next-stage guard: only verified recurring fixed festivals may become features (never generic bank holidays)."""
    return rec.get('status') == VERIFIED_FESTIVAL and bool(rec.get('festival_id'))


def announcement_known_before(rec, issue_date):
    """Announced/special days are usable only when a verified announcement date is on or before the issue date."""
    a = rec.get('announcement_date')
    return bool(a) and dt.date.fromisoformat(str(a)) <= issue_date

## A. Registry, งบคำขอเครือข่าย, อินพุต v1 (อ่านอย่างเดียว) และ metadata ของ catalog ทางการ

- `registry.json` ใต้ external root เก็บ URL, URL ที่ resolve, เวลาดึง/ตรวจ, SHA256, ขนาด, header, สถานะ ของทุกไฟล์; ไฟล์ preflight ของ Codex ลงทะเบียนแยก (`preflight:*`) พร้อม mtime ของแคช และระบุว่าเวลาดาวน์โหลดเดิม **ไม่ทราบ**
- ดึง `package_show` ปัจจุบันของ MOT (roadaccident) และกรมทางหลวง (AADT) เก็บเป็น snapshot ที่ลงทะเบียน hash; ถ้าดึงไม่ได้จึงใช้สำเนา preflight พร้อมบันทึกเป็นช่องว่าง
- ตรวจ SHA256 ของ snapshot ดิบ v1 ทั้ง 6 ไฟล์ก่อนใช้ และตรวจซ้ำตอนจบว่าไม่เปลี่ยน

In [ ]:
# A. Registry, request budget, read-only v1 inputs and official catalog metadata (MOT accidents, DOH traffic)
NET = CFG['http']
BUDGET = RequestBudget(NET['max_network_requests_per_run'])
REG = load_registry(REGISTRY_PATH)
RUN = {'started_utc': utc_now(), 'sections': {}}
GAPS, COVERAGE, DERIVED = [], [], []
(EXT / 'derived').mkdir(parents=True, exist_ok=True)


def save_reg():
    write_json(REGISTRY_PATH, REG)


def fetch(res_id, url, rel, **kw):
    e = try_fetch(res_id, url, rel, EXT, REG, NET, BUDGET, refresh=REFRESH, offline=OFFLINE, **kw)
    save_reg()
    return e


V1MAN = json.loads((V1 / 'data' / 'raw' / 'accidents' / 'source_manifest.json').read_text('utf-8'))
V1_HASHES_BEFORE = verify_v1_snapshot(V1, V1MAN)
panel_path = V1 / 'data' / 'processed' / 'province_day_panel.csv'
PANEL_PROVINCES = (sorted(pd.read_csv(panel_path, usecols=['province'], dtype=str)['province'].dropna().str.strip().unique().tolist())
                   if panel_path.is_file() else [])
if not PANEL_PROVINCES:
    GAPS.append('v1 province_day_panel.csv not found; province names were not compared with the v1 panel')


def catalog(key, res_id):
    """Live package_show snapshot (cached, hash-registered); falls back to the Codex preflight copy only if the live fetch fails."""
    pf = CFG['preflight_files'][key]
    pre = None
    if (PREFLIGHT / pf['file']).is_file():
        pre = register_preflight('preflight:' + key, pf['url'], PREFLIGHT / pf['file'], REG, 'codex_preflight_download_original_time_unknown')
    live = fetch(res_id, pf['url'], f'raw/catalog/{key}.json', validate=json_validator(require_success=True))
    if live.get('sha256'):
        return json.loads((EXT / live['path']).read_text('utf-8')), 'notebook_snapshot', live, pre
    if pre:
        GAPS.append(f'{key}: live package_show unavailable ({live.get("error")}); using the Codex preflight copy')
        return json.loads((PREFLIGHT / pf['file']).read_text('utf-8')), 'codex_preflight', pre, pre
    raise RuntimeError(f'{key}: neither a live catalog snapshot nor the preflight copy is available; check network/allowlist')


MOT_PKG, mot_src, mot_e, mot_pre = catalog('mot_package', 'mot_package_show')
DOH_PKG, doh_src, doh_e, doh_pre = catalog('doh_package', 'doh_package_show')
for key, pkg, src, e, pre in (('mot_package', MOT_PKG, mot_src, mot_e, mot_pre), ('doh_package', DOH_PKG, doh_src, doh_e, doh_pre)):
    r = pkg['result']
    RUN['sections'][key] = {'used': src, 'sha256': e.get('sha256'), 'retrieved_at_utc': e.get('retrieved_at_utc'),
                            'preflight_sha256': (pre or {}).get('sha256'), 'preflight_identical': bool(pre) and pre.get('sha256') == e.get('sha256'),
                            'metadata_modified': r.get('metadata_modified'), 'license': r.get('license_title'), 'n_resources': len(r.get('resources', []))}
    COVERAGE.append({'source_group': 'catalog', 'resource_id': key, 'year': None, 'format': 'ckan_package_show_json',
                     'rows': len(r.get('resources', [])), 'n_columns': None, 'mapped_fields': 'resources[url,format,created,last_modified,size]',
                     'missing_fields': None, 'unmapped_headers': None, 'coverage_start': None, 'coverage_end': None,
                     'status': src, 'sha256': e.get('sha256'), 'note': f"license {r.get('license_title')}; metadata_modified {r.get('metadata_modified')}"})
print('v1 raw snapshot verified (read-only):', len(V1_HASHES_BEFORE), 'files | panel provinces:', len(PANEL_PROVINCES))
print({k: RUN['sections'][k] for k in ('mot_package', 'doh_package')})

## B. หลักฐานวันที่รายปี (Date evidence)

ปัญหาเดิม: ไฟล์ v1 ปี 2021–2023 มีวันที่แบบ `xx/xx/yyyy` ที่อ่านได้ทั้ง D/M และ M/D (v1 กักกันไว้ ทำให้วันที่ 1–12 ของเดือนถูกนับขาด), ปี 2024/2025 เป็น Excel serial, ปี 2026 ไม่มี ACC_CODE

วิธี: อ่าน XLSX ทางการของ MOT (preflight ของ Codex ถ้าเป็นไฟล์ XLSX สมบูรณ์ มิฉะนั้นดาวน์โหลดจาก URL ใน package) เฉพาะเซลล์วันที่ที่เป็นชนิด **date/datetime จริง** แล้วจับคู่กับแถว CSV v1 ด้วย ACC_CODE เดิมแบบตรงตัว (ไม่ fuzzy ไม่อาศัยลำดับแถว) แถวหนึ่ง resolve ได้เมื่อ: ทุกแถว XLSX ของ ID นั้นมีวันที่ typed เดียวกัน, ทุกแถว CSV ของ ID นั้นมีสตริงวันที่เดียวกัน และวันที่ XLSX เป็นหนึ่งในการอ่านที่เป็นไปได้ของแถว CSV นั้นเอง

- ข้อสรุปเป็น **รายปี** (ไม่ใช้กฎเดียวทุกปี) และใช้เพื่อรายงานเท่านั้น การ resolve ทำรายแถว
- ID ซ้ำ/ขัดกัน, ID ที่ไม่มีใน XLSX, เซลล์ที่เป็นข้อความ, วันที่ไม่ตรง → ไม่ resolve (คงการกักกันแบบ v1) พร้อมนับจำนวน
- ID ที่มีเฉพาะใน XLSX (แถวใหม่/แก้ไข) นับไว้แต่ **ไม่เพิ่ม** เป็นเหตุการณ์ และไม่แทนที่ไฟล์ดิบ v1
- ปี 2026 ไม่มี ID จึงทำเพียงโปรไฟล์รูปแบบวันที่ (diagnostic)

In [ ]:
# B. Date evidence by year: v1 raw CSV readings vs TYPED XLSX date cells, exact original ACC_CODE join only
D = CFG['date_evidence']
mot_res = {r['url'].rsplit('/', 1)[-1]: r for r in MOT_PKG['result']['resources'] if r.get('url')}
DATE_SUMMARY, row_frames = [], []
for f in V1MAN['files']:
    ce, be = int(f['ce_year']), int(f['be_year'])
    if ce not in D['years_ce']:
        continue
    csv = pd.read_csv(V1 / f['path'], dtype=str, keep_default_na=False, encoding=f['encoding'])
    assert len(csv) == f['rows'], f['file']
    rec = {'year_ce': ce, 'year_be': be, 'csv_file': f['file'], 'csv_sha256': f['sha256'], 'csv_rows': len(csv),
           'csv_has_acc_code': D['id_header'] in csv.columns, **date_format_profile(csv[D['date_header']], ce)}
    if ce in D['diagnostic_only_years_ce']:
        rec['use_policy'] = 'diagnostic_only_no_acc_code_not_a_label'
    elif ce in D['xlsx_years_ce']:
        name = D['xlsx_name_pattern'].format(year=ce)
        res = mot_res.get(name)
        url = res['url'] if res else None
        xpath, xsrc, xe = None, None, {}
        pre = PREFLIGHT / name
        if pre.is_file():
            xe = register_preflight(f'preflight:mot_accident_{ce}_xlsx', url, pre, REG, 'codex_preflight_download_original_time_unknown',
                                    meta={'catalog': resource_meta(res) if res else None})
            if zipfile.is_zipfile(pre):
                xpath, xsrc = pre, 'codex_preflight'
            else:
                xe['last_status'] = 'preflight_not_a_complete_xlsx'
                GAPS.append(f'date evidence: preflight {name} is not a complete XLSX (download may have been interrupted); trying the official URL')
        if xpath is None and url:
            xe = fetch(f'mot_accident_{ce}_xlsx', url, f'raw/mot/{name}', validate=xlsx_validator, max_bytes=D['max_xlsx_bytes'],
                       meta={'source_group': 'date_evidence', 'catalog': resource_meta(res)})
            if xe.get('sha256'):
                xpath, xsrc = EXT / xe['path'], 'downloaded_by_notebook09'
        save_reg()
        rec.update(xlsx_name=name, xlsx_url=url, xlsx_source=xsrc, xlsx_sha256=xe.get('sha256'), xlsx_bytes=xe.get('bytes'),
                   xlsx_catalog_size=(res or {}).get('size'), xlsx_catalog_last_modified=(res or {}).get('last_modified'))
        rec['xlsx_size_matches_catalog'] = (xe.get('bytes') == res.get('size')) if res and res.get('size') and xe.get('bytes') else None
        if xpath is None:
            rec['use_policy'] = 'xlsx_unavailable_v1_quarantine_kept'
            GAPS.append(f"date evidence: {name} unavailable ({xe.get('error') or 'not listed in MOT package'}); {ce} ambiguous rows stay quarantined")
        else:
            try:
                x, xmeta = read_xlsx_id_dates(xpath, D['id_header'], D['date_header'], D['max_header_scan_rows'])
            except Exception as ex:  # a corrupt/unexpected workbook is a recorded gap, never a guess
                x, xmeta = None, {'error': f'{type(ex).__name__}: {ex}'}
            rec['xlsx_meta'] = xmeta
            if x is None:
                rec['use_policy'] = 'xlsx_unreadable_v1_quarantine_kept'
                GAPS.append(f"date evidence: {name} unreadable: {xmeta['error']}")
            else:
                csv_rows = pd.DataFrame({'source_row': np.arange(2, len(csv) + 2), 'acc_code': csv[D['id_header']], 'date_raw': csv[D['date_header']]})
                rows, summ = audit_dates_by_id(csv_rows, x, ce)
                rows.insert(0, 'source_year_be', be)
                rows.insert(0, 'source_year_ce', ce)
                row_frames.append(rows)
                rec.update(summ)
                rec['use_policy'] = 'resolve_only_rows_with_unique_typed_id_date_agreement'
    else:
        rec['use_policy'] = ('csv_dates_unambiguous_no_resolution_needed' if rec['csv_ambiguous_slash_rows'] == 0
                             else 'ambiguous_rows_without_xlsx_evidence_kept_quarantined')
    DATE_SUMMARY.append(rec)
    print(f"  {ce}: {rec['csv_format_summary']} | policy={rec['use_policy']} | resolved={rec.get('n_resolved', '-')} "
          f"ambiguous_resolved={rec.get('n_ambiguous_resolved', '-')}/{rec['csv_ambiguous_slash_rows']} mismatch={rec.get('n_mismatch', '-')} "
          f"conclusion={rec.get('conclusion', '-')}", flush=True)
DATE_ROWS = pd.concat(row_frames, ignore_index=True) if row_frames else pd.DataFrame(columns=['source_year_ce', 'resolution_status', 'resolved_date'])
DATE_ROWS.to_csv(EXT / 'derived' / 'date_resolution_by_id.csv', index=False)
DERIVED.append('derived/date_resolution_by_id.csv')
dd = CFG['mot']['data_dictionary']
if dd.get('url'):
    de = fetch('mot_data_dictionary', dd['url'], 'raw/mot/data_dictionary' + Path(urlparse(dd['url']).path).suffix, validate=nonempty_validator)
    dd_status = de.get('last_status') + ('' if de.get('sha256') else f": {de.get('error')}")
else:
    dd_status = dd['status']
GAPS.append(f'MOT data dictionary: {dd_status}; typed XLSX date cells are the primary evidence')
write_json(REP / 'date_evidence_by_year.json', {'policy': D['policy'], 'data_dictionary_status': dd_status,
                                                 'per_row_file': 'derived/date_resolution_by_id.csv (under the external root; not in git)',
                                                 'per_row_sha256': sha256_file(EXT / 'derived' / 'date_resolution_by_id.csv'), 'years': DATE_SUMMARY})
write_csv(REP / 'date_evidence_by_year.csv', flatten_records(DATE_SUMMARY))
RUN['sections']['date_evidence'] = {'data_dictionary_status': dd_status, 'years': [r['year_ce'] for r in DATE_SUMMARY]}
pd.DataFrame(DATE_SUMMARY)[['year_ce', 'csv_rows', 'csv_ambiguous_slash_rows', 'use_policy'] +
                           [c for c in ('n_resolved', 'n_ambiguous_resolved', 'n_mismatch', 'conclusion') if c in pd.DataFrame(DATE_SUMMARY)]]

## C. ปริมาณจราจร (AADT กรมทางหลวง)

- เลือก resource CSV รายปีสำรวจ พ.ศ. 2563–2568 จาก package ทางการของกรมทางหลวง (ถ้ามีหลายไฟล์ในปีเดียวกันเลือกตัวที่ `created` ล่าสุดและบันทึกตัวอื่นไว้) เก็บไฟล์ดิบไว้ตามเดิม แล้ว normalize เป็น ปี/จังหวัด/สายทาง/ตอนควบคุม/จุดสำรวจ/จำนวนรถแต่ละประเภท/รวม/สัดส่วนรถหนัก
- ตรวจ schema รายปี, ค่าตัวเลข (ไม่เติม 0), key ซ้ำ (สายทาง+ตอนควบคุม+จุดสำรวจ), ผลรวมประเภทรถเทียบคอลัมน์ "รวม" และสัดส่วนรถหนัก
- หน่วย: คัน/วัน ณ จุดสำรวจ **รวมกันเป็นจำนวนรถจริงหรือจราจร real-time ไม่ได้** ขั้นถัดไปใช้ได้เพียง mean/median/coverage รายจังหวัด
- บันทึก metadata ของแต่ละ resource (created/last_modified/metadata_modified/release) เพื่อใช้ตัดสินการ join: โมเดลใช้ได้เฉพาะปีสำรวจที่เผยแพร่ก่อนเวลาออกพยากรณ์ (timestamp ของ catalog เป็นเพียง proxy) และห้ามใช้ค่าปีเป้าหมายเองเป็น feature ต้นปี
- เทียบไฟล์ preflight `aadt_2021.csv` กับไฟล์ที่ notebook ดาวน์โหลดเองด้วย SHA256

In [ ]:
# C. Traffic: official DOH annual AADT survey CSVs (BE 2563-2568 = CE 2020-2025 where listed in the package)
T = CFG['traffic']
selected = select_aadt_resources(DOH_PKG['result']['resources'], T['survey_years_be'], T['prefer_format'])
meta_rows, norm_frames = [], []
for be in T['survey_years_be']:
    res, alts = selected.get(be, (None, []))
    row = {'survey_year_be': be, 'survey_year_ce': be - 543, 'normalize_status': 'not_listed_in_package', 'rows': None,
           'duplicate_key_rows': None, 'total_mismatch_rows': None, 'catalog_created': None}
    if res is None:
        GAPS.append(f'traffic: no {T["prefer_format"]} AADT resource for BE {be} in the DOH package')
        meta_rows.append(row)
        continue
    row.update(resource_meta(res))
    row['alternative_resources'] = ';'.join(a.get('url', '') for a in alts)
    e = fetch(f'doh_aadt_{be}_csv', res['url'], 'raw/doh_aadt/' + res['url'].rsplit('/', 1)[-1], validate=nonempty_validator,
              meta={'source_group': 'traffic', 'catalog': resource_meta(res)})
    row.update(fetch_status=e.get('last_status'), sha256=e.get('sha256'), bytes=e.get('bytes'), error=e.get('error'))
    row['size_matches_catalog'] = (e.get('bytes') == res.get('size')) if e.get('bytes') and res.get('size') else None
    if not e.get('sha256'):
        row['normalize_status'] = 'unavailable'
        GAPS.append(f"traffic: BE {be} AADT download unavailable: {e.get('error')}")
        meta_rows.append(row)
        continue
    raw, enc = read_csv_bytes((EXT / e['path']).read_bytes(), T['encodings_to_try'])
    norm, rep = normalize_aadt(raw, T['aliases'], T['required_fields'], T['total_components'], T['heavy_components'], be)
    row.update(encoding=enc, rows=len(raw), normalize_status=rep['status'], duplicate_key_rows=rep['duplicate_key_rows'],
               total_mismatch_rows=rep['total_check'].get('n_mismatch_gt1'), heavy_share_mismatch_rows=rep['heavy_check'].get('n_mismatch_gt0_05pp'),
               negative_values=rep['negative_values'], nonnumeric_count_cells=sum(rep['nonnumeric_cells'].values()),
               provinces=int(norm['province'].nunique()), provinces_not_in_v1_panel=';'.join(sorted(set(norm['province']) - set(PANEL_PROVINCES))) if PANEL_PROVINCES else None,
               total_aadt_median=float(norm['total_aadt'].median()) if norm['total_aadt'].notna().any() else None)
    if rep['status'] == 'normalized':
        norm['source_sha256'] = e['sha256']
        norm['catalog_created'] = row['catalog_created']
        norm_frames.append(norm)
    else:
        GAPS.append(f"traffic: BE {be} schema incomplete, missing {rep['missing_fields']}")
    COVERAGE.append({'source_group': 'traffic', 'resource_id': res.get('id'), 'year': be, 'format': res.get('format'), 'rows': len(raw),
                     'n_columns': raw.shape[1], 'mapped_fields': ','.join(f'{k}={v}' for k, v in rep['mapped'].items()),
                     'missing_fields': ','.join(rep['missing_fields']), 'unmapped_headers': ','.join(rep['unmapped_text_headers']),
                     'coverage_start': f'{be - 543}-01-01', 'coverage_end': f'{be - 543}-12-31', 'status': rep['status'], 'sha256': e['sha256'],
                     'note': f"unit vehicles/day at survey points; encoding {enc}; survey-year coverage assumed from title (BE {be})"})
    print(f"  AADT BE {be}: {rep['status']} rows={len(raw)} dup_keys={rep['duplicate_key_rows']} total_check={rep['total_check']} enc={enc}")
    meta_rows.append(row)
TRAFFIC_META = pd.DataFrame(meta_rows)
pf = CFG['preflight_files']['doh_aadt_2564_csv']
pe = register_preflight('preflight:doh_aadt_2564_csv', pf['url'], PREFLIGHT / pf['file'], REG, 'codex_preflight_download_original_time_unknown')
save_reg()
fetched_2564 = TRAFFIC_META.loc[TRAFFIC_META['survey_year_be'] == 2564, 'sha256']
pe_match = (not fetched_2564.empty) and fetched_2564.iloc[0] == pe['sha256']
RUN['sections']['traffic'] = {'preflight_2021_csv_matches_notebook_download': bool(pe_match), 'package_metadata_modified': DOH_PKG['result'].get('metadata_modified'),
                              'license': DOH_PKG['result'].get('license_title'), 'unit_note': T['unit_note'], 'availability_policy': T['availability_policy']}
if not pe_match:
    GAPS.append('traffic: Codex preflight aadt_2021.csv does not match (or could not be compared with) the notebook download of the BE2564 CSV')
TRAFFIC_NORM = pd.concat(norm_frames, ignore_index=True) if norm_frames else pd.DataFrame()
if len(TRAFFIC_NORM):
    TRAFFIC_NORM.to_csv(EXT / 'derived' / 'traffic_aadt_normalized.csv', index=False)
    DERIVED.append('derived/traffic_aadt_normalized.csv')
write_csv(REP / 'traffic_resource_metadata.csv', TRAFFIC_META)
GAPS.append('traffic: catalog created/modified timestamps are publication proxies, not proof the current bytes existed then')
print('preflight 2021 CSV identical to notebook download:', pe_match)
TRAFFIC_META[['survey_year_be', 'normalize_status', 'rows', 'duplicate_key_rows', 'total_mismatch_rows', 'catalog_created']]

## D. ภูมิศาสตร์และจุดสอบถามอากาศรายจังหวัด

- ขอบเขต ADM1 จาก geoBoundaries API ปัจจุบัน แต่ดาวน์โหลดไฟล์ GeoJSON จาก URL ที่ **pin commit** ไว้ใน config (บันทึกว่า URL ปัจจุบันของ API ตรงกับที่ pin หรือไม่) — แหล่งนี้มาจาก OSM/ODbL **ไม่ใช่** ข้อมูลทางการของรัฐไทย
- ตารางรหัส/ชื่อจังหวัดไทย–อังกฤษจาก thailand-geography-json (บันทึก license และ commit ล่าสุดที่ API ตอบ ณ เวลาดึง)
- จับคู่ shape → จังหวัดด้วย ISO `TH-xx` = รหัสจังหวัด ก่อน ถ้าไม่มีจึงใช้ชื่ออังกฤษที่ normalize แล้ว **ตรงตัว** เท่านั้น ไม่เดาแบบ fuzzy; ที่จับคู่ไม่ได้แสดงชัดเจน
- จุด = `representative_point()` ของ shapely ซึ่งรับประกันว่าอยู่ **ภายใน** รูปจังหวัด (ตรวจ `contains` ซ้ำ) เป็นจุด proxy สำหรับสอบถามอากาศ ไม่ใช่ตำแหน่งถนน/เหตุการณ์ และไม่ใช้พิกัดจากอุบัติเหตุ
- เทียบชื่อไทยกับ 77 จังหวัดใน panel v1 (อ่านอย่างเดียว)

In [ ]:
# D. Geography: geoBoundaries ADM1 (pinned release) + maintained province code table -> representative query points
G = CFG['geography']
POINTS = pd.DataFrame(columns=POINT_COLUMNS)
GEO_STATUS = {'summary': 'geography not collected'}
api_e = fetch('geoboundaries_api_tha_adm1', G['geoboundaries_api'], 'raw/geo/geoboundaries_api_THA_ADM1.json', validate=json_validator())
api_meta = {}
if api_e.get('sha256'):
    a = json.loads((EXT / api_e['path']).read_text('utf-8'))
    a = a[0] if isinstance(a, list) and a else a
    api_meta = {k: a.get(k) for k in ('boundaryID', 'boundaryISO', 'boundaryType', 'boundaryYearRepresented', 'buildDate', 'sourceDataUpdateDate',
                                      'boundarySource', 'boundaryLicense', 'licenseDetail', 'licenseSource', 'admUnitCount', 'gjDownloadURL')}
api_meta['pinned_gj_download_url'] = G['pinned_gj_download_url']
api_meta['current_api_url_matches_pinned'] = api_meta.get('gjDownloadURL') == G['pinned_gj_download_url']
gj_e = fetch('geoboundaries_tha_adm1_geojson_pinned', G['pinned_gj_download_url'], 'raw/geo/geoBoundaries-THA-ADM1.geojson',
             validate=json_validator(), max_bytes=G['max_geojson_bytes'])
pv_e = fetch('thailand_geography_provinces_json', G['province_table_url'], 'raw/geo/provinces.json', validate=json_validator())
repo_e = fetch('thailand_geography_repo_api', G['province_table_repo_api'], 'raw/geo/thailand_geography_repo.json', validate=json_validator())
commit_e = fetch('thailand_geography_provinces_commit_api', G['province_table_commit_api'], 'raw/geo/thailand_geography_provinces_commit.json',
                 validate=json_validator())
table_meta = {}
if repo_e.get('sha256'):
    r = json.loads((EXT / repo_e['path']).read_text('utf-8'))
    table_meta['license'] = (r.get('license') or {}).get('spdx_id')
    table_meta['default_branch'] = r.get('default_branch')
if commit_e.get('sha256'):
    c = json.loads((EXT / commit_e['path']).read_text('utf-8'))
    if isinstance(c, list) and c:
        table_meta['latest_commit_touching_file_at_fetch'] = c[0].get('sha')
        table_meta['latest_commit_date'] = ((c[0].get('commit') or {}).get('committer') or {}).get('date')
table_meta['revision_note'] = 'raw URL follows the main branch; commit info is the API answer at its own fetch time, not proof of the cached bytes'
if gj_e.get('sha256') and pv_e.get('sha256'):
    gj = json.loads((EXT / gj_e['path']).read_text('utf-8'))
    shapes = representative_points(gj['features'])
    prov, keys = province_table(json.loads((EXT / pv_e['path']).read_text('utf-8')))
    j = join_shapes(shapes, prov)
    pts_all = shapes.merge(j, on='shape_index').merge(prov, on='province_code', how='left')
    pts_all['in_v1_panel'] = pts_all['name_th'].isin(PANEL_PROVINCES)
    pts_all['geometry_source'] = 'geoBoundaries gbOpen THA ADM1 @ ' + G['pinned_gj_download_url']
    pts_all['geometry_sha256'] = gj_e['sha256']
    pts_all['province_table_sha256'] = pv_e['sha256']
    pts_all['point_method'] = 'shapely_representative_point'
    pts_all['point_meaning'] = 'province label proxy query point; not a road/event location'
    POINTS = pts_all[POINT_COLUMNS].sort_values(['province_code', 'shape_index'], na_position='last').reset_index(drop=True)
    unresolved = POINTS[POINTS['province_code'].isna()]
    missing_panel = sorted(set(PANEL_PROVINCES) - set(POINTS['name_th'].dropna()))
    extra = sorted(set(POINTS['name_th'].dropna()) - set(PANEL_PROVINCES)) if PANEL_PROVINCES else []
    GEO_STATUS = {'summary': (f"shapes {len(shapes)} (expected {G['expected_provinces']}); joined by ISO code "
                              f"{int((POINTS['join_method'] == 'iso_3166_2_code').sum())}, by exact normalized English name "
                              f"{int((POINTS['join_method'] == 'exact_normalized_english_name').sum())}, unresolved {len(unresolved)}; "
                              f"points inside polygon {int(POINTS['point_within_polygon'].astype(bool).sum())}; "
                              f"v1 panel provinces without a point: {missing_panel or 'none'}; point provinces not in panel: {extra or 'none'}"),
                  'api_metadata': api_meta, 'province_table': {**keys, **table_meta}, 'unresolved_shapes': unresolved['shape_name'].tolist(),
                  'panel_provinces_missing': missing_panel, 'points_not_in_panel': extra}
    if len(unresolved) or missing_panel or len(shapes) != G['expected_provinces']:
        GAPS.append('geography join incomplete: ' + GEO_STATUS['summary'])
    COVERAGE.append({'source_group': 'geography', 'resource_id': 'geoboundaries_tha_adm1_geojson_pinned', 'year': api_meta.get('boundaryYearRepresented'),
                     'format': 'geojson', 'rows': len(shapes), 'n_columns': None, 'mapped_fields': 'shapeName,shapeISO,shapeID,geometry',
                     'missing_fields': None, 'unmapped_headers': None, 'coverage_start': None, 'coverage_end': None,
                     'status': 'ok', 'sha256': gj_e['sha256'], 'note': G['source_quality_label']})
    COVERAGE.append({'source_group': 'geography', 'resource_id': 'thailand_geography_provinces_json', 'year': None, 'format': 'json',
                     'rows': len(prov), 'n_columns': 3, 'mapped_fields': ','.join(f'{k}={v}' for k, v in keys.items()), 'missing_fields': None,
                     'unmapped_headers': None, 'coverage_start': None, 'coverage_end': None, 'status': 'ok', 'sha256': pv_e['sha256'],
                     'note': f"license {table_meta.get('license')}; {table_meta['revision_note']}"})
else:
    GAPS.append('geography unavailable (geojson or province table missing): weather collection is blocked')
    GEO_STATUS['api_metadata'] = api_meta
write_csv(REP / 'province_points.csv', POINTS)
RUN['sections']['geography'] = GEO_STATUS
print(GEO_STATUS['summary'])
POINTS.head()

## E. พยากรณ์อากาศย้อนหลังที่เก็บถาวร (Open-Meteo Previous Runs API, JMA GSM)

- ขอเฉพาะ 2 ตัวแปร `temperature_2m_previous_day2`, `precipitation_previous_day2` ล็อกโมเดล `jma_gsm` (ไม่ใช้ best-match ที่เปลี่ยนโมเดลอัตโนมัติ) เขตเวลา Asia/Bangkok ปี 2021–2025 สำหรับจุดตัวแทนของ 77 จังหวัด แบ่ง chunk = 1 ปี × 5 จุด
- timeout 120 วินาที, retry สูงสุด 3 ครั้ง, เว้นระยะระหว่างคำขอ, จำกัดจำนวนคำขอต่อรอบ; โดน HTTP 429 (ไม่มี Retry-After สั้น) จะ **หยุด** และบันทึกความครอบคลุมบางส่วน รันซ้ำเพื่อเก็บต่อ (chunk ที่เสร็จแล้วใช้ซ้ำเมื่อ SHA256 ตรง)
- ไม่ใช้ ERA5/ค่าสังเกตการณ์ของวันเป้าหมายแทน; ค่าว่างคงเป็นค่าว่าง
- `nominal_issue_bound_utc = valid_time_utc − 48 ชม.`; สมมติ buffer เผยแพร่ 6 ชม.; แถวรวมใช้ได้เมื่อ `latest_issue_bound + 6 ชม. ≤ (วันเป้าหมาย − 1) 18:00 น.` — ระยะ lead ของ API ไม่ได้พิสูจน์ความหน่วงการส่งข้อมูลจริงในอดีต
- รวมรายวัน และราย 6 ชม. (00–06/06–12/12–18/18–24): `rain_sum`, `temp_mean` คำนวณเฉพาะเมื่อครบทุกชั่วโมง (24 หรือ 6) มิฉะนั้นเป็นค่าว่างพร้อมธง missing; บันทึกพิกัดที่ขอเทียบกับพิกัด grid ที่ได้

เซลล์นี้อาจใช้เวลาหลายนาทีและพิมพ์ความคืบหน้าทุก chunk

In [ ]:
# E. Archived forecast weather: Open-Meteo Previous Runs API, model locked to jma_gsm, lead previous_day2 (~48 h)
W = CFG['weather']
NET_W = dict(NET, timeout_seconds=W['timeout_seconds'], max_retries=W['max_retries'], max_response_bytes=W['max_response_bytes'])
WBUDGET = RequestBudget(W['max_requests_per_run'], parent=BUDGET)
register_preflight('preflight:weather_jma_2021_probe', CFG['preflight_files']['weather_probe']['url'],
                   PREFLIGHT / CFG['preflight_files']['weather_probe']['file'], REG, 'codex_preflight_probe_original_time_unknown')
save_reg()
pts = POINTS[POINTS['province_code'].notna() & POINTS['point_within_polygon'].astype(bool)].sort_values('province_code').reset_index(drop=True)
batches = [pts.iloc[i:i + int(W['locations_per_request'])] for i in range(0, len(pts), int(W['locations_per_request']))]
plan = [(y, bi, b) for y in W['years'] for bi, b in enumerate(batches)]
print(f'weather plan: {len(pts)} province points x {len(W["years"])} years -> {len(plan)} chunks; '
      f'approx. Open-Meteo call weight per chunk {estimate_call_weight(int(W["locations_per_request"]), 366, len(W["hourly_variables"]))} (estimate)')
frames, chunk_log, stop_reason, last_req, fails = [], [], None, 0.0, 0
for y, bi, b in plan:
    rid = f'weather_jma_gsm_{y}_b{bi:02d}'
    rel = f'raw/weather/jma_gsm_{y}_b{bi:02d}.json'
    url = weather_url(W['base_url'], b, f'{y}-01-01', f'{y}-12-31', W)
    e0 = REG['resources'].get(rid) or {}
    cached = bool(e0.get('sha256')) and (EXT / e0['path']).is_file() if e0.get('path') else False
    rec = {'chunk': rid, 'year': y, 'batch': bi, 'province_codes': ','.join(str(int(c)) for c in b['province_code']), 'url': url}
    if stop_reason and not cached:
        rec['status'] = 'not_attempted_' + stop_reason
        chunk_log.append(rec)
        continue
    if not cached and not OFFLINE:
        wait = float(W['min_seconds_between_requests']) - (time.time() - last_req)
        if wait > 0:
            time.sleep(wait)
    try:
        e = cached_fetch(rid, url, rel, EXT, REG, NET_W, WBUDGET, refresh=REFRESH, offline=OFFLINE,
                         validate=weather_validator(len(b)), meta={'source_group': 'weather', 'model': W['model']})
        if not cached:
            last_req = time.time()
        payload = json.loads((EXT / e['path']).read_text('utf-8'))
        f = normalize_weather_payload(payload, b, W['hourly_variables'], W['model'], W['timezone'], CFG['utc_offset_seconds'], W['lead_hours'])
        frames.append(f)
        fails = 0
        rec.update(status=e['last_status'], sha256=e['sha256'], rows=len(f),
                   null_temp=int(f[W['temp_var']].isna().sum()), null_rain=int(f[W['rain_var']].isna().sum()))
    except (RateLimited, BudgetExhausted) as ex:
        last_req = time.time()
        stop_reason = 'rate_limited' if isinstance(ex, RateLimited) else 'request_budget_exhausted'
        rec.update(status=stop_reason, error=str(ex))
    except (FetchError, ValueError, KeyError) as ex:
        last_req = time.time()
        fails += 1
        rec.update(status='failed', error=f'{type(ex).__name__}: {ex}')
        if fails >= int(W['consecutive_failures_to_stop']):
            stop_reason = 'repeated_failures'
    save_reg()
    chunk_log.append(rec)
    print(f"  [{len(chunk_log)}/{len(plan)}] {y} batch {bi:02d} ({len(b)} provinces): {rec['status']} "
          f"rows={rec.get('rows', 0)} null_temp={rec.get('null_temp', '-')} null_rain={rec.get('null_rain', '-')}", flush=True)

W_HOURLY = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
W_DAILY, W_6H = pd.DataFrame(), pd.DataFrame()
cov = []
if len(W_HOURLY):
    hourly_rel = write_table(EXT / 'derived' / 'weather_hourly_jma_gsm', W_HOURLY)
    DERIVED.append(Path(hourly_rel).relative_to(EXT).as_posix())
    agg_args = dict(temp_col=W['temp_var'], rain_col=W['rain_var'], buffer_hours=W['assumed_publication_buffer_hours'],
                    issue_hour_local=CFG['forecast_issue']['issue_hour_local'], utc_offset_seconds=CFG['utc_offset_seconds'])
    W_DAILY = aggregate_weather(W_HOURLY, 24, **agg_args)
    W_6H = aggregate_weather(W_HOURLY, 6, **agg_args)
    for name, df in (('weather_daily_jma_gsm.csv.gz', W_DAILY), ('weather_6h_jma_gsm.csv.gz', W_6H)):
        df.to_csv(EXT / 'derived' / name, index=False)
        DERIVED.append('derived/' + name)
for y in W['years']:
    exp_hours = len(pd.date_range(f'{y}-01-01 00:00', f'{y}-12-31 23:00', freq='h'))
    for r in pts.itertuples():
        sub = W_HOURLY[(W_HOURLY['province_code'] == r.province_code) & (W_HOURLY['valid_time_local'].dt.year == y)] if len(W_HOURLY) else W_HOURLY
        n = len(sub)
        cov.append({'year': y, 'province_code': int(r.province_code), 'name_en': r.name_en, 'requested_lat': r.lat, 'requested_lon': r.lon,
                    'grid_lat': sub['grid_lat'].iloc[0] if n else None, 'grid_lon': sub['grid_lon'].iloc[0] if n else None,
                    'n_expected_hours': exp_hours, 'n_returned_hours': n,
                    'n_valid_temp': int(sub[W['temp_var']].notna().sum()) if n else 0,
                    'n_valid_rain': int(sub[W['rain_var']].notna().sum()) if n else 0,
                    'status': ('complete' if n and int(sub[W['temp_var']].notna().sum()) == exp_hours and int(sub[W['rain_var']].notna().sum()) == exp_hours
                               else ('partial_or_null' if n else 'not_collected'))})
W_COVERAGE = pd.DataFrame(cov)
write_csv(REP / 'weather_coverage.csv', W_COVERAGE)
write_csv(REP / 'weather_chunks.csv', pd.DataFrame(chunk_log))
n_ok = sum(1 for c in chunk_log if c.get('sha256'))
n_complete = int((W_COVERAGE['status'] == 'complete').sum()) if len(W_COVERAGE) else 0
W_STATUS = {
    'readiness_status': 'blocked' if n_ok == 0 else ('collected' if n_complete == len(W_COVERAGE) else 'partial'),
    'requests_used': WBUDGET.used, 'stop_reason': stop_reason, 'chunks_ok': n_ok, 'chunks_total': len(plan),
    'summary': (f'JMA GSM previous_day2 chunks with data: {n_ok}/{len(plan)}; province-years complete (all hours non-null): '
                f'{n_complete}/{len(W_COVERAGE)}; stop reason: {stop_reason or "none"}; '
                f'daily rows {len(W_DAILY)} (complete rain {int(W_DAILY["rain_sum"].notna().sum()) if len(W_DAILY) else 0}); '
                f'issue bound = valid - {W["lead_hours"]} h, buffer {W["assumed_publication_buffer_hours"]} h, all issue_bound_ok='
                f'{bool(W_DAILY["issue_bound_ok"].all()) if len(W_DAILY) else None}'),
}
if len(W_DAILY):
    s = W_DAILY.iloc[[0, len(W_DAILY) // 2, len(W_DAILY) - 1]][['province_code', 'date', 'latest_issue_bound_utc', 'issue_time_utc', 'issue_bound_ok']]
    W_STATUS['sample_lead_bounds'] = s.to_dict('records')
if n_complete < len(W_COVERAGE):
    GAPS.append(f'weather coverage incomplete: {n_complete}/{len(W_COVERAGE)} province-years complete (stop reason: {stop_reason or "null/missing values"}); rerun to resume')
GAPS.append('weather: Open-Meteo previous_day2 lead is nominal; historical public delivery latency is not verified (6 h buffer assumed)')
COVERAGE.append({'source_group': 'weather', 'resource_id': 'open_meteo_previous_runs_jma_gsm', 'year': ','.join(map(str, W['years'])),
                 'format': 'json', 'rows': len(W_HOURLY), 'n_columns': len(W_HOURLY.columns), 'mapped_fields': ','.join(W['hourly_variables']),
                 'missing_fields': None, 'unmapped_headers': None,
                 'coverage_start': str(W_HOURLY['valid_time_local'].min()) if len(W_HOURLY) else None,
                 'coverage_end': str(W_HOURLY['valid_time_local'].max()) if len(W_HOURLY) else None,
                 'status': W_STATUS['readiness_status'], 'sha256': None, 'note': f'model locked to {W["model"]}; tz {W["timezone"]}'})
RUN['sections']['weather'] = {k: v for k, v in W_STATUS.items()}
print(W_STATUS['summary'])

## F. ปฏิทินวันหยุด (ธนาคารแห่งประเทศไทย)

ดาวน์โหลดประกาศวันหยุดสถาบันการเงินของ ธปท. (PDF ปี 2025 เป็น URL ทางการที่ทราบ; ปี 2021–2024 ใช้ URL ตามรูปแบบเดียวกันซึ่ง **ยังไม่ยืนยัน** ถ้า 404 จะบันทึกเป็น unavailable) และหน้าเว็บหลัก/ข่าวประกาศวันหยุดพิเศษ 2025 (27 พ.ย. 2024) ใช้ `pypdf` ดึงข้อความ แล้วหาบรรทัดที่มี "วัน เดือนไทย"

- วันหยุด ธปท. **ไม่ใช่** วันหยุดราชการทั้งหมด (`is_national_public_holiday` ว่างเสมอ)
- ระเบียนที่ถือว่า verified มีเพียงเทศกาลวันที่คงที่ (สงกรานต์ 13–15 เม.ย., 31 ธ.ค./1 ม.ค.) ที่พบคำสำคัญในบรรทัดเดียวกันในเอกสารทางการ ระเบียนอื่นทั้งหมด `excluded_unreviewed_text_extraction`
- วันที่ประกาศ (announcement_date) ใส่เฉพาะเมื่อยืนยันได้ ไม่เดา

In [ ]:
# F. Calendar evidence: Bank of Thailand financial-institution holiday notifications (official PDFs / pages)
C = CFG['calendar']
cal_sources, recs, discovered = [], [], []
for pg in C['pages']:
    e = fetch(pg['id'], pg['url'], f"raw/calendar/{pg['id']}.html", validate=nonempty_validator)
    st = {'source_id': pg['id'], 'url': pg['url'], 'kind': 'html_page', 'status': e.get('last_status'), 'sha256': e.get('sha256'),
          'error': e.get('error'), 'date_mentions': None, 'announcement_date': None, 'announcement_date_status': None}
    if e.get('sha256'):
        html = (EXT / e['path']).read_bytes().decode('utf-8', errors='replace')
        text = html_to_text(html)
        for u in discover_pdf_links(html, pg['url'], NET['allowed_hosts']):
            if u not in discovered:
                discovered.append(u)
        if pg.get('date_text_check'):
            seen = all(t in text for t in pg['date_text_check'])
            st['announcement_date'] = pg['announcement_date'] if seen else None
            st['announcement_date_status'] = 'page_fetched_and_date_text_found' if seen else 'page_fetched_date_text_not_found'
            found = extract_thai_dates(text, pg['announces_for_year'])
            st['date_mentions'] = len(found)
            recs += classify_calendar_records(found, C['recurring_fixed_festivals'], {
                'source_id': pg['id'], 'url': pg['url'], 'sha256': e['sha256'], 'year': pg['announces_for_year'],
                'record_type': 'bot_special_holiday_announcement_text', 'announcement_date': st['announcement_date'],
                'announcement_date_status': st['announcement_date_status']})
    print(f"  page {pg['id']}: {st['status']} {st['error'] or ''}")
    cal_sources.append(st)
pdf_list = [dict(p) for p in C['pdfs']]
known = {p['url'] for p in pdf_list}
for u in discovered[:int(C['max_discovered_pdfs'])]:
    if u not in known:
        pdf_list.append({'id': 'bot_discovered_' + sha256_bytes(u.encode('utf-8'))[:10], 'year': year_from_url(u),
                         'url_basis': 'link_discovered_on_official_page', 'url': u})
for p in pdf_list:
    e = fetch(p['id'], p['url'], f"raw/calendar/{p['id']}.pdf", validate=pdf_validator, max_bytes=C['max_pdf_bytes'])
    st = {'source_id': p['id'], 'url': p['url'], 'kind': 'pdf', 'year': p.get('year'), 'url_basis': p.get('url_basis'),
          'status': e.get('last_status'), 'sha256': e.get('sha256'), 'error': e.get('error'), 'date_mentions': None,
          'announcement_date': None, 'announcement_date_status': 'annual_notification_publication_date_not_verified'}
    if e.get('sha256') and p.get('year'):
        try:
            text, npages = pdf_text((EXT / e['path']).read_bytes())
            found = extract_thai_dates(text, int(p['year']))
            st.update(pages=npages, text_chars=len(text), date_mentions=len(found), parse_status='text_extracted')
            recs += classify_calendar_records(found, C['recurring_fixed_festivals'], {
                'source_id': p['id'], 'url': p['url'], 'sha256': e['sha256'], 'year': int(p['year']),
                'record_type': 'bot_financial_institution_holiday', 'announcement_date': None,
                'announcement_date_status': st['announcement_date_status']})
        except Exception as ex:  # unreadable PDF text is a recorded gap, never an invented calendar
            st['parse_status'] = f'pdf_text_failed: {type(ex).__name__}: {ex}'
    elif e.get('sha256'):
        st['parse_status'] = 'year_unknown_not_parsed'
    print(f"  pdf {p['id']} ({p.get('url_basis')}): {st['status']} mentions={st['date_mentions']} {st['error'] or ''}")
    cal_sources.append(st)
CAL_RECORDS = dedupe_calendar_records(pd.DataFrame(recs, columns=CAL_COLUMNS))
FEST_EVIDENCE = festival_evidence(CAL_RECORDS, C['recurring_fixed_festivals'], C['years'])
write_csv(REP / 'calendar_records.csv', CAL_RECORDS)
write_csv(REP / 'calendar_festival_evidence.csv', FEST_EVIDENCE)
for st in cal_sources:
    COVERAGE.append({'source_group': 'calendar', 'resource_id': st['source_id'], 'year': st.get('year'), 'format': st['kind'],
                     'rows': st.get('date_mentions'), 'n_columns': None, 'mapped_fields': None, 'missing_fields': None,
                     'unmapped_headers': None, 'coverage_start': None, 'coverage_end': None,
                     'status': st['status'] if not st.get('parse_status') else f"{st['status']};{st['parse_status']}",
                     'sha256': st.get('sha256'), 'note': st.get('url_basis') or st.get('announcement_date_status')})
    if st['status'] == 'unavailable':
        GAPS.append(f"calendar source {st['source_id']} unavailable: {st['error']}")
GAPS.append('no official national public-holiday calendar collected; BOT financial-institution holidays are a different, narrower list')
GAPS.append('moveable/lunar and special announced holidays have no verified announcement-date record usable as a feature yet')
n_ver = int(CAL_RECORDS['status'].eq(VERIFIED_FESTIVAL).sum())
CAL_STATUS = {'summary': f"BOT sources fetched: {sum(1 for s in cal_sources if s.get('sha256'))}/{len(cal_sources)}; date-mention records: {len(CAL_RECORDS)} "
                         f"(verified fixed-festival: {n_ver}; excluded unreviewed: {len(CAL_RECORDS) - n_ver}); "
                         f"festival x year confirmed: {int(FEST_EVIDENCE['confirmed_in_official_pdf'].sum())}/{len(FEST_EVIDENCE)}"}
RUN['sections']['calendar'] = {'sources': cal_sources, **CAL_STATUS}
print(CAL_STATUS['summary'])
FEST_EVIDENCE

In [ ]:
# G. Feature-source readiness, schema coverage, Thai audit report and sources manifest
def _status(n_ok, n_total):
    return 'blocked' if n_ok == 0 else ('collected' if n_ok == n_total else 'partial')

fest_ok = int(FEST_EVIDENCE['confirmed_in_official_pdf'].sum()) if len(FEST_EVIDENCE) else 0
traffic_ok = int((TRAFFIC_META['normalize_status'] == 'normalized').sum()) if len(TRAFFIC_META) else 0
evidence = {
    'holiday_festival': (_status(fest_ok, len(FEST_EVIDENCE)) if len(FEST_EVIDENCE) else 'blocked',
                         f'fixed festival x year confirmed in official BOT PDF text: {fest_ok}/{len(FEST_EVIDENCE)}; general public-holiday calendar: not collected'),
    'weather': (W_STATUS['readiness_status'], W_STATUS['summary']),
    'season': ('derived', 'calendar-month proxy computed from the target date in the next task; no external source'),
    'timeblock': ('derived', 'province x 6 h target blocks derived from each event\'s own time in the next task (label side only)'),
    'traffic': (_status(traffic_ok, len(CFG['traffic']['survey_years_be'])),
                f'normalized DOH AADT survey years: {traffic_ok}/{len(CFG["traffic"]["survey_years_be"])}'),
    'history': ('blocked', 'report availability at issue time not verified; conditional research only'),
}
READINESS = pd.DataFrame([{'group': r['group'], 'status': evidence[r['group']][0], 'evidence': evidence[r['group']][1],
                           **{k: r[k] for k in ('next_stage_allowed_use', 'granularity', 'available_at_policy', 'leakage_risk')}}
                          for r in CFG['readiness']])
assert set(READINESS['group']) == {'holiday_festival', 'weather', 'season', 'timeblock', 'traffic', 'history'}
COVERAGE_DF = pd.DataFrame(COVERAGE)
write_csv(REP / 'feature_source_readiness.csv', READINESS)
write_csv(REP / 'external_schema_coverage.csv', COVERAGE_DF)

ds = pd.DataFrame(DATE_SUMMARY)
md = ['# รายงานตรวจแหล่งข้อมูลภายนอก v2 (Notebook 09)', '',
      f'สร้างเมื่อ (UTC): {utc_now()} | สถานะ: งานวิจัยระหว่างดำเนินการ ไม่ใช่ระบบพยากรณ์ที่พร้อมใช้ | ยังไม่มีการเทียบโมเดล', '',
      '## 1. หลักฐานวันที่รายปี (จับคู่ ACC_CODE เดิมแบบตรงตัวกับเซลล์วันที่ typed ใน XLSX ทางการ)', '',
      '| ปี ค.ศ. | แถว CSV v1 | รูปแบบ | แถว resolve ได้ | กำกวมใน v1 | กำกวมที่ resolve ได้ | ไม่ตรง | สรุป | นโยบาย |', '|---|---|---|---|---|---|---|---|---|']
for r in DATE_SUMMARY:
    md.append(f"| {r['year_ce']} | {r['csv_rows']} | {r['csv_format_summary']} | {r.get('n_resolved', '')} | {r['csv_ambiguous_slash_rows']} | "
              f"{r.get('n_ambiguous_resolved', '')} | {r.get('n_mismatch', '')} | {r.get('conclusion', '')} | {r['use_policy']} |")
md += ['', 'ข้อสรุปเป็นรายปี ไม่ใช่กฎรวมทุกปี; แถวที่ไม่ resolve ยังถูกกักกันตาม v1; แถวที่มีเฉพาะใน XLSX ไม่ถูกเพิ่ม; ปี 2026 ไม่มี ID จึงเป็น diagnostic เท่านั้น',
       f"เอกสาร data dictionary ของ MOT: {CFG['mot']['data_dictionary']['status']} (ไม่ใช้ข้อความค้นหาที่แคชไว้เป็นหลักฐาน)", '',
       '## 2. ความพร้อมของแหล่ง feature', '', '| กลุ่ม | สถานะ | หลักฐาน | การใช้ที่อนุญาตในขั้นถัดไป | นโยบาย available-at | ความเสี่ยง leakage |', '|---|---|---|---|---|---|']
for r in READINESS.itertuples():
    md.append(f'| {r.group} | {r.status} | {r.evidence} | {r.next_stage_allowed_use} | {r.available_at_policy} | {r.leakage_risk} |')
md += ['', '## 3. จราจร (DOH AADT)', '', CFG['traffic']['unit_note'], CFG['traffic']['availability_policy'], '']
if len(TRAFFIC_META):
    md += ['| ปีสำรวจ พ.ศ. | สถานะ | แถว | key ซ้ำ | total ไม่ตรงผลรวม | catalog created (proxy) |', '|---|---|---|---|---|---|']
    for r in TRAFFIC_META.itertuples():
        md.append(f'| {r.survey_year_be} | {r.normalize_status} | {r.rows} | {r.duplicate_key_rows} | {r.total_mismatch_rows} | {r.catalog_created} |')
md += ['', '## 4. ภูมิศาสตร์และจุดสอบถามอากาศ', '', CFG['geography']['source_quality_label'], CFG['geography']['point_meaning'], GEO_STATUS['summary'], '',
       '## 5. พยากรณ์อากาศย้อนหลัง (Open-Meteo Previous Runs, JMA GSM)', '', W_STATUS['summary'], CFG['weather']['lead_note'], '',
       '## 6. ปฏิทิน', '', CFG['calendar']['policy'], CAL_STATUS['summary'], '',
       '## 7. ช่องว่างของแหล่งข้อมูล (บันทึกจากการรันจริง)', '']
md += [f'- {g}' for g in GAPS] or ['- ไม่มี']
(REP / 'source_audit_th.md').write_text('\n'.join(md) + '\n', 'utf-8')

fn_src = functions_cell_source(BACKEND / 'notebooks' / '09_external_collection_date_evidence.ipynb')
versions = {'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__}
for mod in ('openpyxl', 'pypdf', 'shapely', 'pyarrow'):
    try:
        versions[mod] = __import__(mod).__version__
    except Exception:
        versions[mod] = None
cached = []
for rid, e in sorted(REG['resources'].items()):
    cached.append({'resource_id': rid, 'path': e.get('path'), 'preflight_path': e.get('preflight_path'), 'sha256': e.get('sha256'),
                   'bytes': e.get('bytes'), 'last_status': e.get('last_status'), 'url': e.get('url'), 'provenance': e.get('provenance')})
for rel in DERIVED:
    cached.append({'resource_id': 'derived:' + rel, 'path': rel, 'sha256': sha256_file(EXT / rel), 'bytes': (EXT / rel).stat().st_size,
                   'last_status': 'derived_by_notebook09', 'provenance': 'derived_by_notebook09'})
outputs = [{'path': p.relative_to(OUT).as_posix(), 'sha256': sha256_file(p), 'bytes': p.stat().st_size}
           for p in sorted(REP.iterdir()) if p.is_file() and p.name != 'sources_manifest.json']
MANIFEST = {
    'producer': '09_external_collection_date_evidence', 'schema': 'roadaware-v2-sources-manifest/1',
    'status': 'completed' if not GAPS else 'completed_with_gaps',
    'status_note': 'collection/audit only; no feature panel, no model fit, no claim of beating baselines',
    'started_utc': RUN['started_utc'], 'finished_utc': utc_now(), 'offline': OFFLINE, 'refresh': REFRESH,
    'functions_cell_sha256': sha256_bytes(fn_src.encode('utf-8')), 'config_sha256': sha256_file(CFG_PATH),
    'config_path': 'config/external_sources_v2.json', 'external_root': str(EXT), 'output_root': str(OUT), 'v1_data_root': str(V1),
    'v1_raw_manifest_sha256': sha256_file(V1 / 'data' / 'raw' / 'accidents' / 'source_manifest.json'),
    'v1_raw_snapshot_sha256': V1_HASHES_BEFORE, 'versions': versions,
    'network': {'requests_used': BUDGET.used, 'max_requests': BUDGET.max_requests, 'weather_requests_used': W_STATUS['requests_used']},
    'sections': RUN['sections'], 'gaps': GAPS, 'cached_artifacts': cached, 'outputs': outputs,
}
write_json(REP / 'sources_manifest.json', MANIFEST)
print(READINESS[['group', 'status', 'evidence']].to_string(index=False))
print('reports ->', REP)

In [ ]:
# Final checks (passing them means the bookkeeping is consistent; it does NOT prove the external data are complete or correct)
def _reject_constant(c):
    raise ValueError('non-finite JSON constant ' + c)

assert verify_v1_snapshot(V1, V1MAN) == V1_HASHES_BEFORE, 'v1 raw snapshot changed during notebook 09'
for p in sorted(REP.iterdir()):
    if p.is_file():
        assert p.stat().st_size < 5 * 1024 * 1024, f'report file too large for reports/ (row-level data belongs under data/): {p}'
        if p.suffix == '.json':
            json.loads(p.read_text('utf-8'), parse_constant=_reject_constant)
for o in MANIFEST['outputs']:
    assert sha256_file(OUT / o['path']) == o['sha256'], 'output changed after manifest: ' + o['path']
for a in MANIFEST['cached_artifacts']:
    if a.get('sha256') and a.get('path'):
        assert sha256_file(EXT / a['path']) == a['sha256'], 'cached artifact changed: ' + a['path']
if len(DATE_ROWS):
    ok = DATE_ROWS['resolution_status'] == 'resolved'
    assert DATE_ROWS.loc[ok, 'resolved_date'].notna().all() and DATE_ROWS.loc[~ok, 'resolved_date'].isna().all()
    assert not DATE_ROWS['source_year_ce'].isin(CFG['date_evidence']['diagnostic_only_years_ce']).any()
if len(POINTS):
    assert POINTS.loc[POINTS['province_code'].notna(), 'point_within_polygon'].astype(bool).all(), 'query point outside its province'
for agg, n in ((W_DAILY, 24), (W_6H, 6)):
    if len(agg):
        assert agg.loc[agg['rain_sum'].notna(), 'n_valid_rain'].eq(n).all(), 'rain sum from incomplete hours'
        assert agg.loc[agg['temp_mean'].notna(), 'n_valid_temp'].eq(n).all(), 'temperature mean from incomplete hours'
        assert agg['issue_bound_ok'].all(), 'forecast hour whose nominal issue bound + buffer is after the issue time'
if len(CAL_RECORDS):
    assert not CAL_RECORDS['is_national_public_holiday'].eq(True).any(), 'bank holidays must not be labelled national public holidays'
print('final checks passed | status:', MANIFEST['status'])
print('network requests used:', MANIFEST['network']['requests_used'], '| gaps:', len(MANIFEST['gaps']))
for g in MANIFEST['gaps']:
    print('  gap:', g)

## ข้อจำกัดและสัญญาสำหรับขั้นถัดไป (v2 feature panel – งานแยกต่างหาก)

- **วันที่:** ใช้ `data/external_v2/derived/date_resolution_by_id.csv` เฉพาะแถว `resolution_status == 'resolved'` (ID เดิมตรงกันและวันที่ typed ใน XLSX เห็นพ้องกันไม่กำกวม) ร่วมกับ hash ใน `sources_manifest.json`; แถวอื่นยังถูกกักกันตาม v1 ห้ามเลือกวันที่เพื่อให้คะแนนดีขึ้น; ปี 2026 (ไม่มี ID) เป็น diagnostic เท่านั้น; ห้ามเพิ่มแถวที่มีเฉพาะใน XLSX
- **จราจร:** `derived/traffic_aadt_normalized.csv` เป็น AADT ราย จุดสำรวจ × ปีสำรวจ ใช้ได้เพียงสรุป mean/median/coverage รายจังหวัด จากปีสำรวจที่ timestamp ของ catalog (proxy) มาก่อนเวลาออกพยากรณ์เท่านั้น ห้ามใช้ค่าปีเป้าหมายเอง ห้ามรวมเป็นจำนวนรถ
- **อากาศ:** `derived/weather_daily_jma_gsm.csv.gz` และ `weather_6h_jma_gsm.csv.gz` ใช้ได้เฉพาะแถวที่ `rain_missing == False`/`temp_missing == False` และ `issue_bound_ok == True`; จุดเป็น proxy ของจังหวัด ไม่ใช่ตำแหน่งถนน; ความครอบคลุมดู `weather_coverage.csv` (อาจไม่ครบหากโดน rate limit – รันซ้ำเพื่อเก็บต่อ)
- **เทศกาล:** ใช้ได้เฉพาะเทศกาลวันที่คงที่ (สงกรานต์ 13–15 เม.ย., สิ้นปี/ปีใหม่) และระยะห่าง/หน้าต่างรอบวันนั้น; ระเบียนอื่นใน `calendar_records.csv` ถูก exclude ชัดเจน; ไม่มีธง "วันหยุดราชการ" ทั่วไป
- **ฤดูกาล/ช่วงเวลา:** ฤดูกาลเป็น proxy จากเดือน (ไม่ใช่ฝนจริง และไม่เท่ากันทั้งประเทศ); ช่วงเวลา 6 ชม. เป็นหน่วยของเป้าหมาย (จังหวัด × ช่วง) ไม่ใช่ feature จากเวลาของเหตุการณ์เดียวกัน
- **ประวัติ (lags):** ยังถูกบล็อกสำหรับการอ้างใช้งานจริงจนกว่าจะพิสูจน์ความพร้อมของรายงาน ณ เวลาออกพยากรณ์
- ทั้งหมดนี้ยังเป็นงานวิจัย ไม่ใช่ระบบพยากรณ์ออนไลน์ที่พร้อมใช้ และยังไม่มีการเทียบโมเดลใน notebook นี้